In [7]:
"""
PetroCMMS Pro — SUTech Industrial Edition
Sharif University of Technology — Industrial Engineering Department
Computerised Maintenance Management System (CMMS)

Key features:
  - Navy/blue professional theme consistent with SCADA/MES systems
  - IE-focused KPIs: MTBF, MTTR, OEE indicators, criticality levels, downtime tracking
  - Performance: lazy rendering, singleton DB connection pool, no redundant redraws
  - Work Order workflow: Open → In Progress → On Hold → Completed → Closed
  - Criticality matrix: Critical / Major / Minor
  - Asset health scoring with left-bar criticality indicators
  - Uniform chart styling across all dashboard and report views
  - PM compliance tracking and overdue alerts
"""

import sqlite3
import datetime
import re
import tkinter as tk
from tkinter import messagebox, ttk
import customtkinter as ctk
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure
import numpy as np

# ─── Theme ────────────────────────────────────────────────────────────────────
ctk.set_appearance_mode("Light")
ctk.set_default_color_theme("blue")

DB_NAME = "petrocmms.db"

# ─── Color Palette (Industrial Dark) ──────────────────────────────────────────
C = {
    "bg":         "#F0F4F8",   # cool blue-white page background
    "panel":      "#FFFFFF",   # white card surface
    "card":       "#FFFFFF",   # table row bg
    "card_alt":   "#F7FAFC",   # alternating row — barely-there blue
    "sidebar":    "#1E3A5F",   # deep navy sidebar
    "border":     "#D1DCE8",   # soft blue-grey border
    "accent":     "#0057B8",   # strong royal blue accent
    "accent2":    "#00897B",   # teal accent
    "success":    "#2E7D32",   # forest green
    "danger":     "#C62828",   # rich red
    "warning":    "#E65100",   # deep orange
    "muted":      "#78909C",   # blue-grey muted
    "text":       "#1A2B3C",   # dark navy text
    "text2":      "#546E7A",   # secondary text
    "header_bg":  "#1E3A5F",   # same as sidebar — navy table headers
    # chart palette — 6 distinct, accessible, pleasant colours
    "ch1":        "#0057B8",   # royal blue
    "ch2":        "#00897B",   # teal
    "ch3":        "#E65100",   # deep orange
    "ch4":        "#6A1B9A",   # purple
    "ch5":        "#2E7D32",   # green
    "ch6":        "#C62828",   # red
}

PRIORITY_COLORS = {
    "Critical": "#C62828",
    "High":     "#E65100",
    "Medium":   "#0057B8",
    "Low":      "#2E7D32",
}
STATUS_COLORS = {
    "Open":          "#0057B8",
    "In Progress":   "#E65100",
    "On Hold":       "#6A1B9A",
    "Completed":     "#2E7D32",
    "Closed":        "#546E7A",
    "Cancelled":     "#C62828",
    "Operational":   "#2E7D32",
    "Under Repair":  "#E65100",
    "Out of Service":"#C62828",
    "Standby":       "#6A1B9A",
}

# ─── Database Layer ────────────────────────────────────────────────────────────
_conn = None

def get_conn():
    """Singleton connection — avoids repeated open/close overhead."""
    global _conn
    if _conn is None:
        _conn = sqlite3.connect(DB_NAME, check_same_thread=False)
        _conn.row_factory = sqlite3.Row
        _conn.execute("PRAGMA journal_mode=WAL")
        _conn.execute("PRAGMA synchronous=NORMAL")
    return _conn

def fetch_all(query, params=()):
    cur = get_conn().execute(query, params)
    return [tuple(r) for r in cur.fetchall()]

def run_query(query, params=()):
    conn = get_conn()
    conn.execute(query, params)
    conn.commit()

def run_query_lastid(query, params=()):
    conn = get_conn()
    cur = conn.execute(query, params)
    conn.commit()
    return cur.lastrowid

def init_db():
    conn = get_conn()
    conn.executescript('''
        CREATE TABLE IF NOT EXISTS assets (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            asset_code TEXT UNIQUE NOT NULL,
            name TEXT NOT NULL,
            location TEXT,
            area TEXT,
            category TEXT,
            criticality TEXT DEFAULT 'Minor',
            manufacturer TEXT,
            model TEXT,
            serial_number TEXT,
            install_date TEXT,
            purchase_date TEXT,
            warranty_expiry TEXT,
            status TEXT DEFAULT 'Operational',
            operating_hours REAL DEFAULT 0,
            last_failure_date TEXT,
            mtbf_hours REAL,
            mttr_hours REAL,
            description TEXT
        );

        CREATE TABLE IF NOT EXISTS pm_schedules (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            asset_id INTEGER NOT NULL,
            task_name TEXT NOT NULL,
            frequency_type TEXT,
            frequency_value INTEGER,
            last_completion_date TEXT,
            next_due_date TEXT,
            estimated_duration_hrs REAL,
            assigned_team TEXT,
            procedure TEXT,
            checklist TEXT,
            status TEXT DEFAULT 'Active',
            compliance_target REAL DEFAULT 95.0,
            FOREIGN KEY(asset_id) REFERENCES assets(id) ON DELETE CASCADE
        );

        CREATE TABLE IF NOT EXISTS work_orders (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            wo_number TEXT UNIQUE NOT NULL,
            asset_id INTEGER NOT NULL,
            wo_type TEXT,
            title TEXT NOT NULL,
            description TEXT,
            priority TEXT DEFAULT 'Medium',
            criticality TEXT DEFAULT 'Minor',
            status TEXT DEFAULT 'Open',
            reported_date TEXT,
            scheduled_date TEXT,
            started_date TEXT,
            completed_date TEXT,
            closed_date TEXT,
            reported_by TEXT,
            assigned_to TEXT,
            estimated_hrs REAL,
            actual_hrs REAL,
            downtime_hrs REAL DEFAULT 0,
            labor_cost REAL DEFAULT 0,
            parts_cost REAL DEFAULT 0,
            total_cost REAL GENERATED ALWAYS AS (labor_cost + parts_cost) VIRTUAL,
            failure_mode TEXT,
            root_cause TEXT,
            corrective_action TEXT,
            notes TEXT,
            FOREIGN KEY(asset_id) REFERENCES assets(id) ON DELETE CASCADE
        );

        CREATE TABLE IF NOT EXISTS inventory (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            part_number TEXT UNIQUE NOT NULL,
            part_name TEXT NOT NULL,
            category TEXT,
            quantity INTEGER DEFAULT 0,
            reserved_qty INTEGER DEFAULT 0,
            min_stock INTEGER DEFAULT 0,
            max_stock INTEGER,
            reorder_point INTEGER,
            unit TEXT DEFAULT 'pcs',
            location TEXT,
            bin_number TEXT,
            supplier TEXT,
            lead_time_days INTEGER,
            unit_price REAL DEFAULT 0,
            last_received_date TEXT,
            notes TEXT
        );

        CREATE TABLE IF NOT EXISTS labor (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            employee_id TEXT UNIQUE,
            name TEXT NOT NULL,
            skill TEXT,
            trade TEXT,
            shift TEXT,
            hourly_rate REAL DEFAULT 0,
            certification TEXT,
            available INTEGER DEFAULT 1
        );

        CREATE TABLE IF NOT EXISTS downtime_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            asset_id INTEGER,
            wo_id INTEGER,
            start_time TEXT,
            end_time TEXT,
            duration_hrs REAL,
            cause TEXT,
            FOREIGN KEY(asset_id) REFERENCES assets(id)
        );
    ''')
    conn.commit()

    # ── Seed data ──────────────────────────────────────────────────────────────
    if fetch_all("SELECT COUNT(*) FROM assets")[0][0] == 0:
        today = datetime.date.today()
        assets = [
            ("C-101", "Centrifugal Compressor Train A",    "Unit-1", "Olefin Unit",    "Compressor",     "Critical",   "Dresser-Rand", "BCL-404",  "DR-2021-001", "2018-03-15", "2017-12-01", "2023-12-01", "Operational",    12450, None,                      8760.0, 24.0),
            ("C-102", "Centrifugal Compressor Train B",    "Unit-1", "Olefin Unit",    "Compressor",     "Critical",   "Dresser-Rand", "BCL-404",  "DR-2021-002", "2018-03-15", "2017-12-01", "2023-12-01", "Under Repair",   11200, today.isoformat(),         6000.0, 36.0),
            ("B-201", "Steam Boiler #1 — 20 Bar",          "Util",   "Utilities",      "Boiler",         "Critical",   "IHI Corp",     "SB-2000",  "IHI-0981",    "2015-06-01", "2015-01-10", "2025-06-01", "Operational",    38000, None,                      4380.0, 12.0),
            ("B-202", "Steam Boiler #2 — 10 Bar",          "Util",   "Utilities",      "Boiler",         "Major",      "IHI Corp",     "SB-1000",  "IHI-0982",    "2016-06-01", "2016-01-10", "2026-06-01", "Operational",    29000, None,                      5256.0, 8.0),
            ("P-301", "Centrifugal Pump — Feed",           "Unit-2", "Reaction Unit",  "Pump",           "Major",      "KSB",          "MegaCPK",  "KSB-2019-07", "2019-07-20", "2019-05-01", "2024-07-01", "Operational",    18700, None,                      2920.0, 6.0),
            ("P-302", "Centrifugal Pump — Recycle",        "Unit-2", "Reaction Unit",  "Pump",           "Minor",      "KSB",          "MegaCPK",  "KSB-2019-08", "2019-07-20", "2019-05-01", "2024-07-01", "Standby",        14500, None,                      3650.0, 4.0),
            ("HE-401","Shell & Tube Heat Exchanger — Pre", "Unit-3", "Distillation",   "Heat Exchanger", "Major",      "API Heat",     "BEM-600",  "API-HE-001",  "2016-09-01", "2016-06-01", None,         "Operational",    45000, None,                      8760.0, 16.0),
            ("FW-501", "Fire Water Pump — Diesel",         "FW",     "Safety Systems", "Pump",           "Critical",   "Pentair",      "Aurora",   "PEN-FWP-001", "2017-01-01", "2016-10-01", "2027-01-01", "Operational",    3200,  None,                      8760.0, 2.0),
            ("AG-601", "Air-Cooled Generator 500kVA",      "Util",   "Utilities",      "Electrical",     "Critical",   "Caterpillar",  "C15",      "CAT-2020-05", "2020-05-10", "2020-02-01", "2025-05-01", "Operational",    9800,  None,                      4380.0, 6.0),
            ("T-701",  "Storage Tank — Naphtha 5000m³",   "Tank",   "Tank Farm",      "Tank",           "Minor",      "Local",        "API-650",  "TK-N5000",    "2010-01-01", "2009-06-01", None,         "Operational",    None,  None,                      None,   None),
        ]
        for a in assets:
            run_query(
                "INSERT INTO assets (asset_code,name,location,area,category,criticality,"
                "manufacturer,model,serial_number,install_date,purchase_date,warranty_expiry,"
                "status,operating_hours,last_failure_date,mtbf_hours,mttr_hours) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
                a
            )

        pm_data = [
            (1, "Monthly Lubrication & Vibration Check",  "monthly",    1,  (today+datetime.timedelta(days=5)).isoformat(),  2.0,  "Mechanical Team", "Lubrication procedure MEC-001"),
            (1, "Quarterly Bearing Inspection",           "quarterly",  1,  (today+datetime.timedelta(days=18)).isoformat(), 4.0,  "Mechanical Team", "Bearing check procedure MEC-002"),
            (3, "Weekly Water Quality & Level Check",     "weekly",     1,  (today+datetime.timedelta(days=3)).isoformat(),  1.0,  "Operator",        "Boiler daily ops sheet"),
            (3, "Annual Boiler Overhaul",                 "yearly",     1,  (today+datetime.timedelta(days=120)).isoformat(),48.0, "Overhaul Crew",   "Boiler overhaul SOP-B201"),
            (5, "Monthly Seal & Coupling Inspection",     "monthly",    1,  (today+datetime.timedelta(days=12)).isoformat(), 2.0,  "Mechanical Team", "Pump seal check MEC-005"),
            (8, "Weekly Fire Pump Test Run",              "weekly",     1,  (today+datetime.timedelta(days=2)).isoformat(),  1.0,  "Safety Team",     "NFPA 25 test procedure"),
            (9, "Monthly Generator Load Test",            "monthly",    1,  (today+datetime.timedelta(days=8)).isoformat(),  3.0,  "Electrical Team", "Generator load test EL-001"),
        ]
        for p in pm_data:
            run_query(
                "INSERT INTO pm_schedules (asset_id,task_name,frequency_type,frequency_value,"
                "next_due_date,estimated_duration_hrs,assigned_team,procedure) VALUES (?,?,?,?,?,?,?,?)", p
            )

        wo_data = [
            ("WO-2025-001", 1, "PM", "Monthly Lubrication — C-101",     "High",   "Completed", "2025-04-01", "2025-04-01", "2025-04-01", "R. Ahmad",     "M. Team",   2.0,  2.5,  0.0,  112.5, 0.0),
            ("WO-2025-002", 2, "CM", "Bearing #3 High Vibration Alarm",  "Critical","In Progress","2025-04-15","2025-04-16","2025-04-16","Ops Control",  "M. Team",  12.0, None,  8.0,  0.0,   850.0),
            ("WO-2025-003", 3, "PM", "Boiler Annual Overhaul Prep",      "High",   "Open",      "2025-05-01", "2025-05-10", None,          "Eng. Dept",    "OH Crew",  48.0, None,  0.0,  0.0,   0.0),
            ("WO-2025-004", 5, "CM", "Pump Seal Replacement — P-301",   "Medium", "Completed", "2025-03-20", "2025-03-21", "2025-03-22", "Operator",     "M. Team",   6.0,  7.5,  6.0,  337.5, 420.0),
            ("WO-2025-005", 9, "PM", "Generator Monthly Load Test",      "Medium", "Completed", "2025-04-10", "2025-04-10", "2025-04-10", "Util. Ops",    "E. Team",   3.0,  3.0,  0.0,  135.0, 0.0),
            ("WO-2025-006", 8, "PM", "Fire Pump Weekly Test",            "High",   "Completed", "2025-04-14", "2025-04-14", "2025-04-14", "Safety",       "S. Team",   1.0,  1.0,  0.0,   45.0, 0.0),
            ("WO-2025-007", 2, "EM", "Emergency Seal Failure — C-102",   "Critical","On Hold",  "2025-04-16", "2025-04-16", "2025-04-16", "Ops Control",  "M. Team",  16.0, None, 36.0, 200.0, 1200.0),
            ("WO-2025-008", 7, "PM", "HE-401 Tube Bundle Cleaning",     "Low",    "Open",      "2025-05-15", "2025-05-20", None,          "Eng. Dept",    "OH Crew",   8.0, None,  0.0,  0.0,   0.0),
        ]
        for w in wo_data:
            run_query(
                "INSERT INTO work_orders (wo_number,asset_id,wo_type,title,priority,status,"
                "reported_date,scheduled_date,completed_date,reported_by,assigned_to,"
                "estimated_hrs,actual_hrs,downtime_hrs,labor_cost,parts_cost) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)", w
            )

        inv_data = [
            ("BRG-6312-2RS", "Deep Groove Ball Bearing 6312-2RS", "Bearings",      24, 5,  5,  30,  "Central Store", "A-03-12",  42.50, 14),
            ("BRG-6309-2RS", "Deep Groove Ball Bearing 6309-2RS", "Bearings",       8, 5,  3,  20,  "Central Store", "A-03-14",  38.00, 14),
            ("SEAL-MEC-25",  "Mechanical Seal 25mm Shaft",         "Seals",          6, 3,  2,  12,  "Seal Store",    "B-01-05", 185.00, 21),
            ("SEAL-MEC-40",  "Mechanical Seal 40mm Shaft",         "Seals",          2, 3,  2,  10,  "Seal Store",    "B-01-06", 220.00, 21),
            ("OIL-ISO46",    "Turbine Oil ISO VG 46 (20L drum)",   "Lubricants",    35, 10, 8,  60,  "Oil Store",     "C-02-01",  28.00,  7),
            ("OIL-ISO68",    "Compressor Oil ISO VG 68 (20L drum)","Lubricants",    18, 10, 5,  50,  "Oil Store",     "C-02-02",  32.00,  7),
            ("GASK-SPIRO-6", "Spiral Wound Gasket DN150 300#",     "Gaskets",        4, 5,  2,  20,  "Gasket Store",  "B-03-08",  18.50, 10),
            ("FILT-AIR-12",  "Air Filter Cartridge 12\" HEPA",     "Filters",        6, 4,  3,  18,  "Filter Store",  "D-01-03",  65.00, 14),
            ("VBELT-B80",    "V-Belt Type B80",                    "Belts",         12, 6,  4,  24,  "Central Store", "A-05-02",   8.50,  7),
            ("COUP-FLEX-4",  "Flexible Coupling Insert Size 4",    "Couplings",      3, 2,  2,   8,  "Central Store", "A-06-01",  95.00, 21),
        ]
        for inv in inv_data:
            run_query(
                "INSERT INTO inventory (part_number,part_name,category,quantity,min_stock,"
                "reserved_qty,max_stock,location,bin_number,unit_price,lead_time_days) VALUES (?,?,?,?,?,?,?,?,?,?,?)", inv
            )

        labor_data = [
            ("EMP-001", "Ahmed Al-Rashid",   "Senior Mechanic",    "Mechanical",  "Day",   55.0, "Piping, Rotating Equipment"),
            ("EMP-002", "Reza Ahmadi",       "Mechanical Technician","Mechanical", "Day",   42.0, "Pump & Compressor"),
            ("EMP-003", "Saeed Karimi",      "Electrician IV",     "Electrical",  "Day",   48.0, "LV/MV Switchgear, Instruments"),
            ("EMP-004", "Youssef Hassan",    "Instrument Tech",    "I&C",         "Day",   50.0, "DCS, PLC, Field Instruments"),
            ("EMP-005", "Mehdi Norouzi",     "Boiler Operator",    "Operations",  "Shift", 38.0, "Boiler, Steam Systems"),
            ("EMP-006", "Khalid Al-Mutairi", "Safety Tech",        "HSE",         "Day",   44.0, "LOTO, Fire Systems, Permits"),
            ("EMP-007", "Faris Al-Zahrani",  "Mechanical Technician","Mechanical", "Night", 42.0, "Pumps, Valves"),
            ("EMP-008", "Ibrahim Qasim",     "Electrician III",    "Electrical",  "Night", 44.0, "Motors, Cables"),
        ]
        for lab in labor_data:
            run_query(
                "INSERT INTO labor (employee_id,name,skill,trade,shift,hourly_rate,certification) VALUES (?,?,?,?,?,?,?)", lab
            )


# ═══════════════════════════════════════════════════════════════════════════════
#  MAIN APPLICATION
# ═══════════════════════════════════════════════════════════════════════════════
class CMMSPro:
    def __init__(self, root: ctk.CTk):
        self.root = root
        self.root.title("PetroCMMS Pro  |  SUTech — Industrial Engineering")
        self.root.geometry("1440x860")
        self.root.minsize(1200, 700)
        self.root.configure(fg_color=C["bg"])

        # Fonts — Dubai Medium throughout
        # Dubai is built-in on Windows 10+ and macOS 10.14+.
        # tkinter accepts the family name; "normal" weight in Dubai = Medium cut.
        _F = "Dubai"
        self.F = {
            "h1":   ctk.CTkFont(_F, 20, "bold"),
            "h2":   ctk.CTkFont(_F, 15, "bold"),
            "h3":   ctk.CTkFont(_F, 13, "bold"),
            "body": ctk.CTkFont(_F, 12, "normal"),
            "mono": ctk.CTkFont(_F, 12, "normal"),
            "tag":  ctk.CTkFont(_F, 10, "bold"),
            "nav":  ctk.CTkFont(_F, 13, "bold"),
            "big":  ctk.CTkFont(_F, 34, "bold"),
        }

        self._page_cache = {}     # avoid rebuilding unchanged pages
        self._current_page = None
        self._selected_ids = {}   # per-section selected row id

        self.root.grid_columnconfigure(1, weight=1)
        self.root.grid_rowconfigure(0, weight=1)

        self._build_sidebar()
        self._build_main_area()
        self.show_dashboard()
        self._check_alerts()

    # ── Sidebar ────────────────────────────────────────────────────────────────
    def _build_sidebar(self):
        sb = ctk.CTkFrame(self.root, width=230, corner_radius=0, fg_color=C["sidebar"],
                          border_width=1, border_color=C["border"])
        sb.grid(row=0, column=0, sticky="nsew")
        sb.grid_propagate(False)

        # ── Logo / Branding ───────────────────────────────────────────────────
        logo = ctk.CTkFrame(sb, fg_color="transparent")
        logo.pack(fill="x", padx=16, pady=(24, 2))
        ctk.CTkLabel(logo, text="⚙", font=ctk.CTkFont("Dubai", 28),
                     text_color="#FFFFFF").pack(side="left")
        ctk.CTkLabel(logo, text=" PetroCMMS", font=self.F["h2"],
                     text_color="#FFFFFF").pack(side="left")
        ctk.CTkLabel(sb, text="  Industrial Edition", font=self.F["tag"],
                     text_color="#90CAF9").pack(anchor="w", padx=16, pady=(0, 2))

        # SUTech attribution line
        sutech_row = ctk.CTkFrame(sb, fg_color="#162D4A", corner_radius=6)
        sutech_row.pack(fill="x", padx=10, pady=(2, 14))
        ctk.CTkLabel(sutech_row, text="SUTech  ·  Dept. of Industrial Engineering",
                     font=self.F["tag"], text_color="#64B5F6").pack(
                     anchor="w", padx=10, pady=5)

        # Divider
        ctk.CTkFrame(sb, height=1, fg_color=C["border"]).pack(fill="x", padx=10, pady=4)

        nav_items = [
            ("⬛", "DASHBOARD",   self.show_dashboard),
            ("◈", "ASSETS",       self.show_assets),
            ("◷", "PM SCHEDULES", self.show_pm),
            ("▣", "WORK ORDERS",  self.show_wo),
            ("▦", "INVENTORY",    self.show_inventory),
            ("◉", "LABOR",        self.show_labor),
            ("⬡", "REPORTS",      self.show_reports),
        ]

        self._nav_btns = {}
        for icon, label, cmd in nav_items:
            btn = ctk.CTkButton(
                sb,
                text=f"  {icon}  {label}",
                font=self.F["nav"],
                fg_color="transparent",
                anchor="w", height=42,
                corner_radius=8,
                text_color="#B0C4DE",
                hover_color="#2A4F7A",
            )
            btn.configure(command=lambda c=cmd, b=btn, l=label: self._nav_click(c, b, l))
            btn.pack(fill="x", padx=10, pady=2)
            self._nav_btns[label] = btn

        # Status bar at bottom
        ctk.CTkFrame(sb, height=1, fg_color=C["border"]).pack(fill="x", padx=10, pady=4, side="bottom")
        status = ctk.CTkFrame(sb, fg_color="transparent")
        status.pack(side="bottom", fill="x", padx=16, pady=12)
        ctk.CTkLabel(status, text="● SYSTEM ONLINE", font=self.F["tag"],
                     text_color="#A5D6A7").pack(side="left")
        ctk.CTkLabel(status, text=datetime.date.today().strftime("%d %b %Y"),
                     font=self.F["tag"], text_color="#B0BEC5").pack(side="right")

    def _nav_click(self, cmd, btn, label):
        # Highlight active nav
        for lbl, b in self._nav_btns.items():
            b.configure(fg_color="transparent", text_color="#B0C4DE")
        btn.configure(fg_color="#2A4F7A", text_color="#FFFFFF")
        cmd()

    def _highlight_nav(self, label):
        for lbl, b in self._nav_btns.items():
            if lbl == label:
                b.configure(fg_color="#2A4F7A", text_color="#FFFFFF")
            else:
                b.configure(fg_color="transparent", text_color="#B0C4DE")

    # ── Main area ──────────────────────────────────────────────────────────────
    def _build_main_area(self):
        self._main = ctk.CTkFrame(self.root, corner_radius=0,
                                   fg_color=C["bg"], border_width=0)
        self._main.grid(row=0, column=1, sticky="nsew")
        self._main.grid_rowconfigure(0, weight=1)
        self._main.grid_rowconfigure(1, weight=0)
        self._main.grid_columnconfigure(0, weight=1)
        self._content = None

        # Footer
        footer = ctk.CTkFrame(self._main, fg_color=C["sidebar"], corner_radius=0, height=22)
        footer.grid(row=1, column=0, sticky="ew")
        footer.grid_propagate(False)
        ctk.CTkLabel(footer,
                     text="  PetroCMMS Pro  ·  SUTech  ·  Dept. of Industrial Engineering",
                     font=self.F["tag"], text_color="#90CAF9").pack(side="left", padx=10)
        ctk.CTkLabel(footer, text=datetime.date.today().strftime("%d %B %Y  "),
                     font=self.F["tag"], text_color="#B0BEC5").pack(side="right")

    def _set_page(self, builder_fn, label):
        """Destroy previous page, build new one. No animation overhead."""
        if self._content:
            self._content.destroy()
        self._content = ctk.CTkScrollableFrame(
            self._main,
            fg_color=C["bg"],
            scrollbar_button_color=C["border"],
            scrollbar_button_hover_color=C["accent"],
            corner_radius=0,
        )
        self._content.grid(row=0, column=0, sticky="nsew", padx=0, pady=0)
        self._content.grid_columnconfigure(0, weight=1)
        self._highlight_nav(label)
        builder_fn(self._content)

    # ── Alert Check ────────────────────────────────────────────────────────────
    def _check_alerts(self):
        today = datetime.date.today().isoformat()
        overdue = fetch_all(
            "SELECT wo_number, title, priority FROM work_orders "
            "WHERE scheduled_date < ? AND status NOT IN ('Completed','Closed','Cancelled')",
            (today,)
        )
        if overdue:
            msgs = [f"[{p}] {n} — {t}" for n, t, p in overdue]
            messagebox.showwarning(
                "⚠  Overdue Work Orders",
                f"{len(overdue)} work order(s) past scheduled date:\n\n" + "\n".join(msgs[:8])
            )

    # ══════════════════════════════════════════════════════════════════════════
    #  SHARED UI COMPONENTS
    # ══════════════════════════════════════════════════════════════════════════

    def _page_header(self, parent, title, subtitle=""):
        hdr = ctk.CTkFrame(parent, fg_color="transparent")
        hdr.pack(fill="x", padx=24, pady=(20, 0))
        ctk.CTkLabel(hdr, text=title, font=self.F["h1"],
                     text_color=C["sidebar"]).pack(side="left")
        if subtitle:
            ctk.CTkLabel(hdr, text=f"  ·  {subtitle}", font=self.F["body"],
                         text_color=C["muted"]).pack(side="left", padx=4, pady=6)
        # SUTech stamp — right side of header
        ctk.CTkLabel(hdr, text="SUTech · PetroCMMS Pro",
                     font=self.F["tag"], text_color=C["muted"]).pack(side="right", padx=4)
        # Accent line
        ctk.CTkFrame(parent, height=2, fg_color=C["accent"],
                     corner_radius=2).pack(fill="x", padx=24, pady=(4, 14))

    def _kpi_card(self, parent, title, value, unit="", color=None, col=0):
        color = color or C["accent"]
        card = ctk.CTkFrame(parent, fg_color=C["card"], corner_radius=12,
                             border_width=1, border_color=C["border"])
        card.pack(side="left", expand=True, fill="both", padx=6, pady=4)
        ctk.CTkLabel(card, text=title, font=self.F["tag"],
                     text_color=C["muted"]).pack(anchor="w", padx=14, pady=(14, 2))
        val_frame = ctk.CTkFrame(card, fg_color="transparent")
        val_frame.pack(anchor="w", padx=14, pady=(0, 14))
        # Accent bar on left
        ctk.CTkFrame(card, width=3, fg_color=color,
                     corner_radius=2).place(relx=0, rely=0, relheight=1)
        ctk.CTkLabel(val_frame, text=str(value), font=self.F["big"],
                     text_color=color).pack(side="left")
        if unit:
            ctk.CTkLabel(val_frame, text=f" {unit}", font=self.F["body"],
                         text_color=C["muted"]).pack(side="left", pady=10)

    def _section_card(self, parent, title="", col=0, row=0, colspan=1, rowspan=1):
        card = ctk.CTkFrame(parent, fg_color=C["panel"], corner_radius=12,
                             border_width=1, border_color=C["border"])
        if title:
            ctk.CTkLabel(card, text=title, font=self.F["h3"],
                         text_color=C["sidebar"]).pack(anchor="w", padx=14, pady=(12, 4))
            ctk.CTkFrame(card, height=1, fg_color=C["border"]).pack(fill="x", padx=10)
        else:
            pass  # no title — no extra padding
        return card

    def _status_badge(self, parent, text):
        color = STATUS_COLORS.get(text, C["muted"])
        frm = ctk.CTkFrame(parent, fg_color=color, corner_radius=6, height=22)
        ctk.CTkLabel(frm, text=text, font=self.F["tag"],
                     text_color="#FFFFFF").pack(padx=8, pady=2)
        return frm

    def _priority_badge(self, parent, text):
        color = PRIORITY_COLORS.get(text, C["muted"])
        frm = ctk.CTkFrame(parent, fg_color=color, corner_radius=6, height=22)
        ctk.CTkLabel(frm, text=text, font=self.F["tag"],
                     text_color="#FFFFFF").pack(padx=8, pady=2)
        return frm

    def _build_table(self, parent, headers, rows, on_select=None,
                     col_widths=None, highlight_col=None, highlight_map=None,
                     compact=False):
        """
        High-performance table using ttk.Treeview with custom dark styling.
        Much faster than CTkFrame grid tables for large datasets.
        """
        style = ttk.Style()
        style.theme_use("clam")
        style.configure("CMMS.Treeview",
            background="#FFFFFF",
            fieldbackground="#FFFFFF",
            foreground=C["text"],
            rowheight=32,
            borderwidth=0,
            relief="flat",
            font=("Dubai", 11),
        )
        style.configure("CMMS.Treeview.Heading",
            background=C["header_bg"],
            foreground="#FFFFFF",
            font=("Dubai", 11, "bold"),
            relief="flat",
            borderwidth=0,
            padding=(8, 6),
        )
        style.map("CMMS.Treeview",
            background=[("selected", "#BBDEFB")],
            foreground=[("selected", C["text"])],
        )
        style.configure("CMSScroll.Vertical.TScrollbar",
            troughcolor=C["bg"], arrowcolor=C["muted"],
            background=C["border"],
        )

        tree_frame = ctk.CTkFrame(parent, fg_color="transparent")
        # Use compact mode to reduce top padding for PM/WO/Inventory
        pady_val = 0 if compact else 8
        tree_frame.pack(fill="both", expand=True, padx=10, pady=pady_val)

        vsb = ttk.Scrollbar(tree_frame, orient="vertical",
                            style="CMSScroll.Vertical.TScrollbar")
        vsb.pack(side="right", fill="y")

        tree = ttk.Treeview(
            tree_frame,
            columns=headers,
            show="headings",
            style="CMMS.Treeview",
            yscrollcommand=vsb.set,
            selectmode="browse",
        )
        vsb.configure(command=tree.yview)
        tree.pack(fill="both", expand=True)

        # Alternating row tags
        tree.tag_configure("even", background="#FFFFFF")
        tree.tag_configure("odd",  background="#F0F7FF")
        # Status color tags
        for status, color in STATUS_COLORS.items():
            tree.tag_configure(f"status_{status}", foreground=color)
        for pri, color in PRIORITY_COLORS.items():
            tree.tag_configure(f"pri_{pri}", foreground=color)

        for i, h in enumerate(headers):
            w = col_widths[i] if col_widths else 130
            tree.heading(h, text=h, anchor="w")
            tree.column(h, width=w, minwidth=60, anchor="w")

        self._tree_id_map = {}
        for idx, row in enumerate(rows):
            row_id = row[0]
            display = [str(c) if c is not None else "—" for c in row[1:]]
            tags = ["even" if idx % 2 == 0 else "odd"]

            if highlight_col is not None and highlight_map:
                val = display[highlight_col]
                tag = highlight_map.get(val)
                if tag:
                    tags.append(tag)

            iid = tree.insert("", "end", values=display, tags=tags)
            self._tree_id_map[iid] = row_id

        if on_select:
            def _on_click(event):
                sel = tree.selection()
                if sel:
                    row_id = self._tree_id_map.get(sel[0])
                    if row_id is not None:
                        on_select(row_id)
            tree.bind("<<TreeviewSelect>>", _on_click)

        self._current_tree = tree
        return tree

    # Original toolbar (used by Assets and Labor)
    def _toolbar(self, parent, add_fn, edit_fn, delete_fn, extra_btns=None):
        bar = ctk.CTkFrame(parent, fg_color="transparent")
        bar.pack(fill="x", padx=10, pady=(4, 4))

        btn_cfg = dict(font=self.F["h3"], corner_radius=8, height=28, width=110)
        ctk.CTkButton(bar, text="+ ADD",
                      fg_color=C["success"], hover_color="#16A34A", text_color="#FFFFFF",
                      command=add_fn, **btn_cfg).pack(side="left", padx=4)
        ctk.CTkButton(bar, text="✎ EDIT",
                      fg_color=C["warning"], hover_color="#D97706", text_color="#FFFFFF",
                      command=edit_fn, **btn_cfg).pack(side="left", padx=4)
        ctk.CTkButton(bar, text="✕ DELETE",
                      fg_color=C["danger"], hover_color="#DC2626", text_color="#FFFFFF",
                      command=delete_fn, **btn_cfg).pack(side="left", padx=4)

        if extra_btns:
            ctk.CTkFrame(bar, width=2, fg_color=C["border"]).pack(side="left", padx=8, fill="y")
            for label, fn, color in extra_btns:
                ctk.CTkButton(bar, text=label,
                              fg_color=color, hover_color=color, text_color="#FFFFFF",
                              command=fn, **btn_cfg).pack(side="left", padx=4)

    # Compact toolbar (used only for PM, Work Orders, Inventory)
    def _toolbar_compact(self, parent, add_fn, edit_fn, delete_fn, extra_btns=None):
        bar = ctk.CTkFrame(parent, fg_color="transparent", height=30)  # fixed height
        bar.pack(fill="x", padx=10, pady=(0, 0))
        bar.pack_propagate(False)  # prevent the frame from growing

        btn_cfg = dict(font=self.F["h3"], corner_radius=6, height=26, width=110)
        ctk.CTkButton(bar, text="+ ADD",
                      fg_color=C["success"], hover_color="#16A34A", text_color="#FFFFFF",
                      command=add_fn, **btn_cfg).pack(side="left", padx=4)
        ctk.CTkButton(bar, text="✎ EDIT",
                      fg_color=C["warning"], hover_color="#D97706", text_color="#FFFFFF",
                      command=edit_fn, **btn_cfg).pack(side="left", padx=4)
        ctk.CTkButton(bar, text="✕ DELETE",
                      fg_color=C["danger"], hover_color="#DC2626", text_color="#FFFFFF",
                      command=delete_fn, **btn_cfg).pack(side="left", padx=4)

        if extra_btns:
            ctk.CTkFrame(bar, width=2, fg_color=C["border"]).pack(side="left", padx=8, fill="y")
            for label, fn, color in extra_btns:
                ctk.CTkButton(bar, text=label,
                              fg_color=color, hover_color=color, text_color="#FFFFFF",
                              command=fn, **btn_cfg).pack(side="left", padx=4)

    def _get_selected_id(self, section_key):
        """Return selected DB row ID from the current treeview."""
        if not hasattr(self, "_current_tree") or not self._current_tree:
            return None
        sel = self._current_tree.selection()
        if not sel:
            messagebox.showinfo("Select Row", "Please click on a row first.")
            return None
        return self._tree_id_map.get(sel[0])

    # ── Entry Dialog ───────────────────────────────────────────────────────────
    def _dialog(self, title, fields, on_submit, width=560, height=680):
        """
        Generic form dialog.
        field keys: label, key, type (entry/combobox/int/float/text), values, required, default
        """
        dlg = ctk.CTkToplevel(self.root)
        dlg.title(title)
        dlg.geometry(f"{width}x{height}")
        dlg.grab_set()
        dlg.transient(self.root)
        dlg.configure(fg_color=C["panel"])
        dlg.resizable(True, True)

        # Header
        hdr = ctk.CTkFrame(dlg, fg_color=C["sidebar"], corner_radius=0)
        hdr.pack(fill="x")
        ctk.CTkLabel(hdr, text=f"  {title}", font=self.F["h2"],
                     text_color="#FFFFFF").pack(side="left", pady=12, padx=10)

        body = ctk.CTkScrollableFrame(dlg, fg_color="transparent",
                                       scrollbar_button_color=C["border"])
        body.pack(fill="both", expand=True, padx=20, pady=10)
        body.grid_columnconfigure(0, weight=1)

        widgets = {}
        for i, f in enumerate(fields):
            ctk.CTkLabel(body, text=f["label"],
                         font=self.F["h3"], text_color=C["text2"]).grid(
                row=i*2, column=0, sticky="w", pady=(8, 0))

            ftype = f.get("type", "entry")
            default = f.get("default", "")

            if ftype == "combobox":
                w = ctk.CTkComboBox(body, values=f.get("values", []),
                                    font=self.F["body"], state="readonly",
                                    fg_color=C["card"], border_color=C["border"],
                                    button_color=C["accent"], text_color=C["text"],
                                    dropdown_fg_color=C["card"],
                                    dropdown_text_color=C["text"],
                                    height=34)
                w.set(str(default) if default is not None else (f.get("values", [""])[0]))
            elif ftype == "text":
                w = ctk.CTkTextbox(body, height=80, font=self.F["body"],
                                   fg_color=C["card"], border_color=C["border"],
                                   text_color=C["text"])
                if default:
                    w.insert("1.0", str(default))
            else:  # entry / int / float
                w = ctk.CTkEntry(body, font=self.F["body"],
                                 fg_color=C["card"], border_color=C["border"],
                                 text_color=C["text"], height=34,
                                 placeholder_text=f.get("placeholder", ""))
                if default is not None and str(default):
                    w.insert(0, str(default))

            w.grid(row=i*2+1, column=0, sticky="ew", pady=(2, 0))
            widgets[f["key"]] = (w, f)

        # Buttons
        btn_row = ctk.CTkFrame(dlg, fg_color="transparent")
        btn_row.pack(fill="x", padx=20, pady=12)
        btn_cfg = dict(font=self.F["h3"], height=36, corner_radius=8)

        def _submit():
            vals = {}
            for key, (w, f) in widgets.items():
                ftype = f.get("type", "entry")
                if ftype == "text":
                    v = w.get("1.0", "end").strip()
                elif ftype == "combobox":
                    v = w.get()
                else:
                    v = w.get().strip()

                if f.get("required") and not v:
                    messagebox.showerror("Required Field", f"{f['label']} is required.")
                    return

                if ftype == "int" and v:
                    try:
                        v = int(v)
                    except ValueError:
                        messagebox.showerror("Validation", f"{f['label']} must be an integer.")
                        return
                elif ftype == "float" and v:
                    try:
                        v = float(v)
                    except ValueError:
                        messagebox.showerror("Validation", f"{f['label']} must be a number.")
                        return

                vals[key] = v if v != "" else None

            try:
                on_submit(vals)
                messagebox.showinfo("Success", "Saved successfully.")
                dlg.destroy()
            except Exception as e:
                messagebox.showerror("Error", str(e))

        ctk.CTkButton(btn_row, text="✔  SAVE", fg_color=C["success"],
                      hover_color="#16A34A", text_color="#FFFFFF",
                      command=_submit, **btn_cfg).pack(side="left", padx=8, fill="x", expand=True)
        ctk.CTkButton(btn_row, text="✕  CANCEL", fg_color=C["danger"],
                      hover_color="#DC2626", text_color="#FFFFFF",
                      command=dlg.destroy, **btn_cfg).pack(side="left", padx=8, fill="x", expand=True)

    def _confirm_delete(self, msg, fn):
        if messagebox.askyesno("⚠ Confirm Delete", msg, icon="warning"):
            fn()

    # ══════════════════════════════════════════════════════════════════════════
    #  DASHBOARD
    # ══════════════════════════════════════════════════════════════════════════
    def show_dashboard(self):
        self._set_page(self._build_dashboard, "DASHBOARD")

    def _build_dashboard(self, frame):
        self._page_header(frame, "DASHBOARD", "Asset Performance Overview")

        # ── Live status banner ────────────────────────────────────────────────
        banner = ctk.CTkFrame(frame, fg_color=C["sidebar"], corner_radius=10, height=38)
        banner.pack(fill="x", padx=18, pady=(0, 8))
        banner.pack_propagate(False)

        # Blinking alert dot for critical WOs
        critical_wos = fetch_all("SELECT COUNT(*) FROM work_orders WHERE priority='Critical' AND status NOT IN ('Completed','Closed','Cancelled')")[0][0]
        dot_color = "#EF4444" if critical_wos > 0 else "#A5D6A7"
        self._dash_dot = ctk.CTkLabel(banner, text="●", font=ctk.CTkFont("Dubai", 14),
                                       text_color=dot_color)
        self._dash_dot.pack(side="left", padx=(14, 4), pady=0)

        alert_txt = f"  {critical_wos} CRITICAL ACTIVE  ·  SYSTEM OPERATIONAL" if critical_wos > 0 else "  ALL SYSTEMS NOMINAL"
        ctk.CTkLabel(banner, text=alert_txt, font=self.F["h3"],
                     text_color="#FFFFFF").pack(side="left")

        # Live clock on right
        self._dash_clock = ctk.CTkLabel(banner, text="", font=self.F["tag"],
                                         text_color="#90CAF9")
        self._dash_clock.pack(side="right", padx=14)
        self._tick_clock(self._dash_clock)

        # Animated pulse on the dot if there are criticals
        if critical_wos > 0:
            self._pulse_dot(self._dash_dot, dot_color, True)

        # ── KPI Row ────────────────────────────────────────────────────────────
        kpi_row = ctk.CTkFrame(frame, fg_color="transparent")
        kpi_row.pack(fill="x", padx=18, pady=4)

        total_assets  = fetch_all("SELECT COUNT(*) FROM assets")[0][0]
        operational   = fetch_all("SELECT COUNT(*) FROM assets WHERE status='Operational'")[0][0]
        open_wos      = fetch_all("SELECT COUNT(*) FROM work_orders WHERE status IN ('Open','In Progress','On Hold')")[0][0]
        low_stock     = fetch_all("SELECT COUNT(*) FROM inventory WHERE quantity <= min_stock")[0][0]
        pm_due_7d     = fetch_all("SELECT COUNT(*) FROM pm_schedules WHERE next_due_date BETWEEN date('now') AND date('now','+7 days')")[0][0]
        total_cost    = fetch_all("SELECT COALESCE(SUM(labor_cost+parts_cost),0) FROM work_orders")[0][0]

        # Health score: % operational of total
        health_pct = round(operational / total_assets * 100) if total_assets else 0
        health_color = C["success"] if health_pct >= 80 else C["warning"] if health_pct >= 60 else C["danger"]

        kpi_items = [
            ("FLEET HEALTH",       f"{health_pct}%",         "",   health_color),
            ("TOTAL ASSETS",       total_assets,              "",   C["text"]),
            ("OPERATIONAL",        operational,               "",   C["success"]),
            ("OPEN WORK ORDERS",   open_wos,                  "",   C["accent2"]),
            ("CRITICAL ACTIVE",    critical_wos,              "",   C["danger"]),
            ("LOW STOCK ITEMS",    low_stock,                 "",   C["warning"]),
            ("PM DUE ≤ 7 DAYS",   pm_due_7d,                 "",   C["accent"]),
            ("TOTAL MAINT. COST",  f"${total_cost:,.0f}",     "",   C["accent2"]),
        ]
        for col, (title, value, unit, color) in enumerate(kpi_items):
            self._kpi_card(kpi_row, title, value, unit, color, col)

        # ── Charts row ────────────────────────────────────────────────────────
        chart_row = ctk.CTkFrame(frame, fg_color="transparent")
        chart_row.pack(fill="x", padx=18, pady=6)

        c1 = self._section_card(chart_row, "WORK ORDER STATUS")
        c1.pack(side="left", expand=True, fill="both", padx=6, pady=6)
        self._chart_wo_status(c1)

        c2 = self._section_card(chart_row, "ASSET CRITICALITY MIX")
        c2.pack(side="left", expand=True, fill="both", padx=6, pady=6)
        self._chart_asset_criticality(c2)

        c3 = self._section_card(chart_row, "DOWNTIME HOURS BY ASSET (TOP 5)")
        c3.pack(side="left", expand=True, fill="both", padx=6, pady=6)
        self._chart_downtime(c3)

        # ── Cost & PM row ─────────────────────────────────────────────────────
        chart_row2 = ctk.CTkFrame(frame, fg_color="transparent")
        chart_row2.pack(fill="x", padx=18, pady=6)

        c4 = self._section_card(chart_row2, "LOW STOCK ITEMS")
        c4.pack(side="left", expand=True, fill="both", padx=6, pady=6)
        self._chart_low_stock(c4)

        c5 = self._section_card(chart_row2, "WO COST BREAKDOWN (LABOR vs PARTS)")
        c5.pack(side="left", expand=True, fill="both", padx=6, pady=6)
        self._chart_cost_breakdown(c5)

        # ── Recent WOs ────────────────────────────────────────────────────────
        rc = self._section_card(frame, "RECENT WORK ORDERS", col=0, row=0, colspan=1)
        rc.pack(fill="x", padx=24, pady=6)
        recent = fetch_all("""
            SELECT wo.id, wo.wo_number, a.asset_code, wo.title, wo.priority, wo.status, wo.scheduled_date
            FROM work_orders wo JOIN assets a ON wo.asset_id=a.id
            ORDER BY wo.id DESC LIMIT 8
        """)
        hdrs = ["WO Number", "Asset", "Title", "Priority", "Status", "Scheduled"]
        self._build_table(rc, hdrs, recent, col_widths=[110, 80, 240, 80, 100, 100],
                          highlight_col=3,
                          highlight_map={s: f"status_{s}" for s in STATUS_COLORS})

        # ── PM Due table ──────────────────────────────────────────────────────
        pm_c = self._section_card(frame, "UPCOMING PM SCHEDULES (30 DAYS)", col=0, row=0, colspan=1)
        pm_c.pack(fill="x", padx=24, pady=6)
        pm_due = fetch_all("""
            SELECT pm.id, a.asset_code, pm.task_name, pm.next_due_date, pm.assigned_team, pm.estimated_duration_hrs
            FROM pm_schedules pm JOIN assets a ON pm.asset_id=a.id
            WHERE pm.next_due_date BETWEEN date('now') AND date('now', '+30 days')
            ORDER BY pm.next_due_date
        """)
        pm_hdrs = ["Asset", "Task Name", "Due Date", "Team", "Est. Hrs"]
        self._build_table(pm_c, pm_hdrs, pm_due, col_widths=[80, 280, 100, 140, 80])

    def _tick_clock(self, label):
        """Update the live clock label every second. Stops if widget is destroyed."""
        try:
            now = datetime.datetime.now().strftime("%H:%M:%S  ·  %d %b %Y")
            label.configure(text=now)
            label.after(1000, lambda: self._tick_clock(label))
        except Exception:
            pass  # widget destroyed when page changes — stop silently

    def _pulse_dot(self, label, color, visible):
        """Blink the alert dot on/off every 700 ms."""
        try:
            label.configure(text_color=color if visible else C["sidebar"])
            label.after(700, lambda: self._pulse_dot(label, color, not visible))
        except Exception:
            pass

    # ── Dashboard Charts ───────────────────────────────────────────────────────
    def _make_fig(self, parent, size=(4, 2.6)):
        fig = Figure(figsize=size, dpi=100, facecolor="#FFFFFF")
        ax = fig.add_subplot(111)
        ax.set_facecolor("#FAFCFF")
        fig.subplots_adjust(left=0.13, right=0.96, top=0.91, bottom=0.22)
        canvas = FigureCanvasTkAgg(fig, master=parent)
        widget = canvas.get_tk_widget()
        widget.configure(bg="#FFFFFF", highlightthickness=0)
        widget.pack(fill="both", expand=True, padx=10, pady=10)
        return fig, ax, canvas

    def _style_ax(self, ax, ylabel="", grid=True):
        """Apply clean, modern light-theme styling to any axes."""
        ax.set_facecolor("#FAFCFF")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_color("#D1DCE8")
        ax.spines["bottom"].set_color("#D1DCE8")
        ax.tick_params(axis="both", colors="#546E7A", labelsize=9, length=3)
        ax.yaxis.label.set_color("#546E7A")
        ax.yaxis.label.set_fontsize(9)
        ax.xaxis.label.set_color("#546E7A")
        ax.xaxis.label.set_fontsize(9)
        if ylabel:
            ax.set_ylabel(ylabel, fontsize=9, color="#546E7A")
        if grid:
            ax.yaxis.grid(True, color="#E8EDF2", linewidth=0.8, linestyle="--", alpha=0.9)
            ax.set_axisbelow(True)
        ax.tick_params(pad=4)

    def _style_legend(self, ax, labels, **kwargs):
        """Uniform legend style matching the chart palette."""
        defaults = dict(
            loc="center left", bbox_to_anchor=(0.97, 0.5),
            fontsize=8.5, frameon=True, framealpha=1,
            edgecolor="#D1DCE8", facecolor="#FFFFFF",
            labelcolor="#1A2B3C", handlelength=1.2, handleheight=1.2,
        )
        defaults.update(kwargs)
        leg = ax.legend(labels, **defaults)
        leg.get_frame().set_linewidth(0.8)
        return leg

    def _chart_wo_status(self, parent):
        data = fetch_all("SELECT status, COUNT(*) FROM work_orders GROUP BY status ORDER BY COUNT(*) DESC")
        if not data:
            ctk.CTkLabel(parent, text="No data yet", font=self.F["body"], text_color=C["muted"]).pack(pady=30)
            return
        labels = [r[0] for r in data]
        sizes  = [r[1] for r in data]
        colors_list = [STATUS_COLORS.get(l, C["muted"]) for l in labels]
        fig, ax, canvas = self._make_fig(parent, (3.8, 2.8))
        fig.patch.set_facecolor("#FFFFFF")
        wedges, texts, autotexts = ax.pie(
            sizes, labels=None, autopct='%1.0f%%', startangle=140,
            colors=colors_list,
            wedgeprops={"linewidth": 2.5, "edgecolor": "#FFFFFF"},
            pctdistance=0.75,
        )
        for at in autotexts:
            at.set_color("#FFFFFF")
            at.set_fontsize(8)
            at.set_fontweight("bold")
        legend_labels = [f"{l}  ({s})" for l, s in zip(labels, sizes)]
        self._style_legend(ax, legend_labels, fontsize=8, handleheight=1.2)
        canvas.draw()

    def _chart_asset_criticality(self, parent):
        data = fetch_all("SELECT criticality, COUNT(*) FROM assets GROUP BY criticality ORDER BY CASE criticality WHEN 'Critical' THEN 1 WHEN 'Major' THEN 2 ELSE 3 END")
        if not data:
            ctk.CTkLabel(parent, text="No data yet", font=self.F["body"], text_color=C["muted"]).pack(pady=30)
            return
        crit_colors = {"Critical": "#C62828", "Major": "#E65100", "Minor": "#2E7D32"}
        labels = [r[0] for r in data]
        sizes  = [r[1] for r in data]
        colors_list = [crit_colors.get(l, C["muted"]) for l in labels]
        fig, ax, canvas = self._make_fig(parent, (3.8, 2.8))
        fig.patch.set_facecolor("#FFFFFF")
        wedges, texts, autotexts = ax.pie(
            sizes, labels=None, autopct='%1.0f%%', startangle=140,
            colors=colors_list,
            wedgeprops={"linewidth": 2.5, "edgecolor": "#FFFFFF"},
            pctdistance=0.72,
        )
        for at in autotexts:
            at.set_color("#FFFFFF")
            at.set_fontsize(9)
            at.set_fontweight("bold")
        legend_labels = [f"{l}  ({s})" for l, s in zip(labels, sizes)]
        self._style_legend(ax, legend_labels, fontsize=8, handleheight=1.2)
        canvas.draw()

    def _chart_downtime(self, parent):
        data = fetch_all("""
            SELECT a.asset_code, COALESCE(SUM(wo.downtime_hrs),0) as dt
            FROM assets a LEFT JOIN work_orders wo ON a.id=wo.asset_id
            GROUP BY a.id ORDER BY dt DESC LIMIT 5
        """)
        if not data:
            ctk.CTkLabel(parent, text="No downtime recorded", font=self.F["body"], text_color=C["muted"]).pack(pady=30)
            return
        names = [r[0] for r in data]
        vals  = [r[1] for r in data]
        # Gradient colour: red → orange → blue by severity
        bar_colors = ["#C62828" if v > 20 else "#E65100" if v > 5 else "#0057B8" for v in vals]
        bar_alphas = [1.0 if v > 0 else 0.35 for v in vals]
        fig, ax, canvas = self._make_fig(parent, (5, 2.8))
        fig.patch.set_facecolor("#FFFFFF")
        bars = ax.bar(names, vals, color=bar_colors, width=0.5, zorder=3,
                      linewidth=0, edgecolor="none")
        for bar, alpha in zip(bars, bar_alphas):
            bar.set_alpha(alpha)
        self._style_ax(ax, ylabel="Downtime (hrs)")
        # Value labels above bars
        for bar, val in zip(bars, vals):
            if val > 0:
                ax.text(bar.get_x() + bar.get_width()/2,
                        bar.get_height() + max(vals)*0.03,
                        f"{val:.0f}h", ha="center", va="bottom",
                        fontsize=9, fontweight="bold", color="#1A2B3C")
        ax.tick_params(axis="x", labelsize=9, rotation=15)
        canvas.draw()

    def _chart_low_stock(self, parent):
        data = fetch_all("SELECT part_name, quantity, min_stock FROM inventory WHERE quantity <= min_stock ORDER BY (quantity - min_stock) LIMIT 6")
        if not data:
            ok_frame = ctk.CTkFrame(parent, fg_color="transparent")
            ok_frame.pack(expand=True)
            ctk.CTkLabel(ok_frame, text="✔", font=ctk.CTkFont("Dubai", 32),
                         text_color="#2E7D32").pack(pady=(20, 4))
            ctk.CTkLabel(ok_frame, text="All stock levels OK", font=self.F["body"],
                         text_color="#2E7D32").pack()
            return
        names = [r[0][:18] for r in data]
        qty   = [r[1] for r in data]
        mins  = [r[2] for r in data]
        fig, ax, canvas = self._make_fig(parent, (4.8, 2.8))
        fig.patch.set_facecolor("#FFFFFF")
        x = np.arange(len(names))
        w = 0.38
        # Min stock as soft background bar
        ax.bar(x, mins, w*2.3, color="#FFF3E0", edgecolor="none", zorder=1, label="_nolegend_")
        # Current stock
        b1 = ax.bar(x - w/2, qty,  w, color="#0057B8", edgecolor="none", zorder=3, label="Current Qty", linewidth=0)
        # Min line
        b2 = ax.bar(x + w/2, mins, w, color="#E65100", edgecolor="none", zorder=3, label="Min Stock",   linewidth=0, alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(names, rotation=22, ha="right", fontsize=8)
        self._style_ax(ax, ylabel="Units")
        self._style_legend(ax, ["Current Qty", "Min Stock"],
                           loc="upper right", bbox_to_anchor=(1.0, 1.0), fontsize=8)
        # Value labels
        for bar in b1:
            h = bar.get_height()
            ax.text(bar.get_x()+bar.get_width()/2, h + max(mins+qty)*0.04,
                    str(int(h)), ha="center", fontsize=8, fontweight="bold", color="#0057B8")
        canvas.draw()

    def _chart_cost_breakdown(self, parent):
        data = fetch_all("SELECT COALESCE(SUM(labor_cost),0), COALESCE(SUM(parts_cost),0) FROM work_orders")
        if not data or (data[0][0] == 0 and data[0][1] == 0):
            ctk.CTkLabel(parent, text="No cost data yet", font=self.F["body"],
                         text_color=C["muted"]).pack(pady=30)
            return
        labor, parts = data[0]
        total = labor + parts
        fig, ax, canvas = self._make_fig(parent, (4.8, 2.8))
        fig.patch.set_facecolor("#FFFFFF")
        categories = ["Labor", "Parts"]
        values     = [labor, parts]
        bar_colors = ["#0057B8", "#00897B"]
        bar_bgs    = ["#E3F2FD", "#E0F2F1"]
        max_val    = total * 1.3 if total > 0 else 1
        # Background track
        ax.barh(categories, [max_val]*2, color=bar_bgs, edgecolor="none", height=0.45, zorder=1)
        # Value bars
        bars = ax.barh(categories, values, color=bar_colors, edgecolor="none", height=0.45, zorder=2)
        ax.set_xlim(0, max_val)
        ax.set_xlabel("USD ($)", fontsize=9, color="#546E7A")
        self._style_ax(ax, grid=False)
        ax.spines["left"].set_visible(False)
        ax.tick_params(left=False)
        for bar, val, col in zip(bars, values, bar_colors):
            pct = val / total * 100 if total > 0 else 0
            ax.text(bar.get_width() + max_val * 0.02,
                    bar.get_y() + bar.get_height()/2,
                    f"${val:,.0f}  ({pct:.0f}%)",
                    va="center", fontsize=9, fontweight="bold", color=col)
        canvas.draw()

    # ══════════════════════════════════════════════════════════════════════════
    #  ASSETS
    # ══════════════════════════════════════════════════════════════════════════
    def show_assets(self):
        self._set_page(self._build_assets, "ASSETS")

    # ── Criticality / Status badge colours ──────────────────────────────────
    CRIT_STYLE = {
        "Critical": ("#FFEBEE", "#C62828", "●"),
        "Major":    ("#FFF3E0", "#E65100", "●"),
        "Minor":    ("#E8F5E9", "#2E7D32", "●"),
    }
    STATUS_STYLE = {
        "Operational":    ("#E8F5E9", "#2E7D32"),
        "Under Repair":   ("#FFF3E0", "#E65100"),
        "Out of Service": ("#FFEBEE", "#C62828"),
        "Standby":        ("#EDE7F6", "#6A1B9A"),
        "Decommissioned": ("#ECEFF1", "#546E7A"),
    }
    CAT_ICON = {
        "Compressor":    "⚙",
        "Boiler":        "🔥",
        "Pump":          "💧",
        "Heat Exchanger":"♨",
        "Tank":          "🛢",
        "Turbine":       "⚡",
        "Electrical":    "⚡",
        "Vessel":        "⬡",
        "Valve":         "🔧",
        "Instrument":    "📡",
        "Other":         "◈",
    }

    def _build_assets(self, frame):
        self._page_header(frame, "ASSET REGISTRY", "Equipment & Machinery Database")

        # ── KPI strip ─────────────────────────────────────────────────────────
        strip = ctk.CTkFrame(frame, fg_color="transparent")
        strip.pack(fill="x", padx=18, pady=4)
        stats = [
            ("CRITICAL ASSETS",    fetch_all("SELECT COUNT(*) FROM assets WHERE criticality='Critical'")[0][0], C["danger"]),
            ("UNDER REPAIR",       fetch_all("SELECT COUNT(*) FROM assets WHERE status='Under Repair'")[0][0],   C["warning"]),
            ("OUT OF SERVICE",     fetch_all("SELECT COUNT(*) FROM assets WHERE status='Out of Service'")[0][0], C["danger"]),
            ("WARRANTY EXPIRED",   fetch_all("SELECT COUNT(*) FROM assets WHERE warranty_expiry < date('now')")[0][0], C["muted"]),
        ]
        for col, (t, v, c) in enumerate(stats):
            self._kpi_card(strip, t, v, color=c, col=col)

        # ── Toolbar ───────────────────────────────────────────────────────────
        def _add():
            self._dialog("ADD ASSET", self._asset_fields(), self._insert_asset)
        def _edit():
            rid = self._asset_selected_id
            if rid:
                row = fetch_all("SELECT asset_code,name,location,area,category,criticality,manufacturer,model,serial_number,install_date,purchase_date,warranty_expiry,status,operating_hours,mtbf_hours,mttr_hours,description FROM assets WHERE id=?", (rid,))
                if row:
                    self._dialog("EDIT ASSET", self._asset_fields(row[0]), lambda v: self._update_asset(rid, v))
            else:
                messagebox.showinfo("Select Row", "Please click on an asset row first.")
        def _delete():
            rid = self._asset_selected_id
            if rid:
                self._confirm_delete("Delete this asset and all linked records?",
                    lambda: [run_query("DELETE FROM assets WHERE id=?", (rid,)), self.show_assets()])
            else:
                messagebox.showinfo("Select Row", "Please click on an asset row first.")

        outer = self._section_card(frame, "")
        outer.pack(fill="both", expand=True, padx=18, pady=6)
        self._toolbar(outer, _add, _edit, _delete)

        # ── Column header row ─────────────────────────────────────────────────
        hdr_row = ctk.CTkFrame(outer, fg_color=C["sidebar"], corner_radius=8, height=36)
        hdr_row.pack(fill="x", padx=10, pady=(6, 0))
        hdr_row.pack_propagate(False)
        col_defs = [
            ("Code",        80,  "w"),
            ("Name",        240, "w"),
            ("Area",        110, "w"),
            ("Category",    120, "w"),
            ("Criticality", 110, "center"),
            ("Status",      130, "center"),
            ("Op. Hours",   90,  "e"),
            ("MTBF (h)",    90,  "e"),
            ("MTTR (h)",    80,  "e"),
        ]
        for col_title, col_w, anchor in col_defs:
            ctk.CTkLabel(hdr_row, text=col_title,
                         font=self.F["h3"], text_color="#FFFFFF",
                         width=col_w, anchor=anchor).pack(side="left", padx=(8,0))

        # ── Scrollable body ───────────────────────────────────────────────────
        body = ctk.CTkScrollableFrame(outer, fg_color="transparent",
                                      scrollbar_button_color=C["border"],
                                      scrollbar_button_hover_color=C["accent"])
        body.pack(fill="both", expand=True, padx=10, pady=(2, 8))

        data = fetch_all("""
            SELECT a.id, a.asset_code, a.name, a.area, a.category, a.criticality,
                   a.status, a.operating_hours, a.mtbf_hours, a.mttr_hours
            FROM assets a ORDER BY
                CASE a.criticality WHEN 'Critical' THEN 1 WHEN 'Major' THEN 2 ELSE 3 END,
                a.asset_code
        """)

        self._asset_selected_id = None
        self._asset_rows = []

        def _make_badge(parent, text, bg, fg, w=100):
            # Styled label only — no filled box/sticker
            ctk.CTkLabel(parent, text=text, font=self.F["tag"],
                         text_color=fg, width=w, anchor="center").pack(side="left", padx=(6, 0))

        def _hrs_label(parent, val, w=80):
            if val is None or val == "":
                txt, col = "—", C["muted"]
            else:
                try:
                    h = float(val)
                    txt = f"{h:,.0f}"
                    col = C["text"]
                except:
                    txt, col = str(val), C["muted"]
            ctk.CTkLabel(parent, text=txt, font=self.F["body"],
                         text_color=col, width=w, anchor="e").pack(side="left", padx=(4,0))

        def _select_row(row_frame, rid):
            # deselect all
            for rf in self._asset_rows:
                rf.configure(fg_color="#FFFFFF" if self._asset_rows.index(rf) % 2 == 0 else "#F0F7FF")
            row_frame.configure(fg_color="#DBEAFE")
            self._asset_selected_id = rid
            # also set the shared _current_tree to None so toolbar uses asset_selected_id
            self._current_tree = None

        for idx, row in enumerate(data):
            rid, code, name, area, cat, crit, status, op_hrs, mtbf, mttr = row
            row_bg = "#FFFFFF" if idx % 2 == 0 else "#F0F7FF"

            rf = ctk.CTkFrame(body, fg_color=row_bg, corner_radius=6, height=44)
            rf.pack(fill="x", pady=2)
            rf.pack_propagate(False)
            self._asset_rows.append(rf)

            # Left accent bar by criticality
            crit_bg, crit_fg, crit_dot = self.CRIT_STYLE.get(crit, ("#F5F5F5","#90A4AE","●"))
            accent_bar = ctk.CTkFrame(rf, fg_color=crit_fg, width=4, corner_radius=0)
            accent_bar.pack(side="left", fill="y", padx=(0,6))

            # Code
            ctk.CTkLabel(rf, text=code, font=self.F["h3"],
                         text_color=C["accent"], width=80, anchor="w").pack(side="left", padx=(2,0))

            # Category icon + Name
            icon = self.CAT_ICON.get(cat, "◈")
            name_frame = ctk.CTkFrame(rf, fg_color="transparent", width=240)
            name_frame.pack(side="left", padx=(2,0))
            name_frame.pack_propagate(False)
            ctk.CTkLabel(name_frame, text=f"{icon}  {name}", font=self.F["body"],
                         text_color=C["text"], anchor="w").pack(fill="both", expand=True, padx=4)

            # Area
            ctk.CTkLabel(rf, text=area or "—", font=self.F["body"],
                         text_color=C["text2"], width=110, anchor="w").pack(side="left", padx=(4,0))

            # Category pill
            cat_pill = ctk.CTkFrame(rf, fg_color="#EEF2FF", corner_radius=6, width=120, height=24)
            cat_pill.pack(side="left", padx=(4,0))
            cat_pill.pack_propagate(False)
            ctk.CTkLabel(cat_pill, text=cat or "—", font=self.F["tag"],
                         text_color="#3730A3").pack(expand=True)

            # Criticality badge (text only, colored)
            _make_badge(rf, f"{crit_dot} {crit}", crit_bg, crit_fg, w=104)

            # Status badge (text only, colored)
            s_bg, s_fg = self.STATUS_STYLE.get(status, ("#F5F5F5","#546E7A"))
            _make_badge(rf, status, s_bg, s_fg, w=124)

            # Op Hours, MTBF, MTTR
            _hrs_label(rf, op_hrs, 90)
            _hrs_label(rf, mtbf,   90)
            _hrs_label(rf, mttr,   80)

            # Click binding — entire row and every child
            def _bind_row(widget, _rf=rf, _rid=rid):
                widget.bind("<Button-1>", lambda e, r=_rf, i=_rid: _select_row(r, i))
                widget.configure(cursor="hand2")
                for child in widget.winfo_children():
                    _bind_row(child)
            _bind_row(rf)

    def _asset_fields(self, d=None):
        d = d or [""]*17
        return [
            {"label": "Asset Code *",       "key": "code",     "type": "entry",    "required": True,  "default": d[0]},
            {"label": "Asset Name *",        "key": "name",     "type": "entry",    "required": True,  "default": d[1]},
            {"label": "Location / Unit",     "key": "loc",      "type": "entry",    "default": d[2]},
            {"label": "Area / Plant",        "key": "area",     "type": "entry",    "default": d[3]},
            {"label": "Category",            "key": "cat",      "type": "combobox", "values": ["Compressor","Boiler","Heat Exchanger","Pump","Tank","Turbine","Electrical","Vessel","Valve","Instrument","Other"], "default": d[4]},
            {"label": "Criticality",         "key": "crit",     "type": "combobox", "values": ["Critical","Major","Minor"], "default": d[5]},
            {"label": "Manufacturer",        "key": "manu",     "type": "entry",    "default": d[6]},
            {"label": "Model",               "key": "model",    "type": "entry",    "default": d[7]},
            {"label": "Serial Number",       "key": "serial",   "type": "entry",    "default": d[8]},
            {"label": "Install Date (YYYY-MM-DD)", "key": "idate", "type": "entry", "default": d[9]},
            {"label": "Purchase Date",       "key": "pdate",    "type": "entry",    "default": d[10]},
            {"label": "Warranty Expiry",     "key": "warr",     "type": "entry",    "default": d[11]},
            {"label": "Status",              "key": "status",   "type": "combobox", "values": ["Operational","Under Repair","Out of Service","Standby","Decommissioned"], "default": d[12]},
            {"label": "Operating Hours",     "key": "ophrs",    "type": "float",    "default": d[13]},
            {"label": "MTBF (hours)",        "key": "mtbf",     "type": "float",    "default": d[14]},
            {"label": "MTTR (hours)",        "key": "mttr",     "type": "float",    "default": d[15]},
            {"label": "Description",         "key": "desc",     "type": "text",     "default": d[16]},
        ]

    def _insert_asset(self, v):
        run_query("INSERT INTO assets (asset_code,name,location,area,category,criticality,manufacturer,model,serial_number,install_date,purchase_date,warranty_expiry,status,operating_hours,mtbf_hours,mttr_hours,description) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
                  (v["code"], v["name"], v["loc"], v["area"], v["cat"], v["crit"],
                   v["manu"], v["model"], v["serial"], v["idate"], v["pdate"],
                   v["warr"], v["status"], v["ophrs"], v["mtbf"], v["mttr"], v["desc"]))
        self.show_assets()

    def _update_asset(self, aid, v):
        run_query("UPDATE assets SET asset_code=?,name=?,location=?,area=?,category=?,criticality=?,manufacturer=?,model=?,serial_number=?,install_date=?,purchase_date=?,warranty_expiry=?,status=?,operating_hours=?,mtbf_hours=?,mttr_hours=?,description=? WHERE id=?",
                  (v["code"], v["name"], v["loc"], v["area"], v["cat"], v["crit"],
                   v["manu"], v["model"], v["serial"], v["idate"], v["pdate"],
                   v["warr"], v["status"], v["ophrs"], v["mtbf"], v["mttr"], v["desc"], aid))
        self.show_assets()

    # ══════════════════════════════════════════════════════════════════════════
    #  PM SCHEDULES (compact toolbar + compact table)
    # ══════════════════════════════════════════════════════════════════════════
    def show_pm(self):
        self._set_page(self._build_pm, "PM SCHEDULES")

    def _build_pm(self, frame):
        self._page_header(frame, "PM SCHEDULES", "Preventive Maintenance Program")

        strip = ctk.CTkFrame(frame, fg_color="transparent")
        strip.pack(fill="x", padx=18, pady=4)
        overdue = fetch_all("SELECT COUNT(*) FROM pm_schedules WHERE next_due_date < date('now')")[0][0]
        due7    = fetch_all("SELECT COUNT(*) FROM pm_schedules WHERE next_due_date BETWEEN date('now') AND date('now','+7 days')")[0][0]
        due30   = fetch_all("SELECT COUNT(*) FROM pm_schedules WHERE next_due_date BETWEEN date('now') AND date('now','+30 days')")[0][0]
        self._kpi_card(strip, "OVERDUE PM",      overdue, color=C["danger"],  col=0)
        self._kpi_card(strip, "DUE ≤ 7 DAYS",   due7,    color=C["warning"], col=1)
        self._kpi_card(strip, "DUE ≤ 30 DAYS",  due30,   color=C["accent"],  col=2)

        def _add():
            assets = fetch_all("SELECT id, asset_code, name FROM assets ORDER BY asset_code")
            ac = [f"{a[1]} — {a[2]} (ID:{a[0]})" for a in assets]
            self._dialog("ADD PM SCHEDULE", self._pm_fields(asset_choices=ac), self._insert_pm, height=620)
        def _edit():
            rid = self._get_selected_id("pm")
            if rid:
                row = fetch_all("SELECT asset_id,task_name,frequency_type,frequency_value,last_completion_date,next_due_date,estimated_duration_hrs,assigned_team,procedure,checklist,status,compliance_target FROM pm_schedules WHERE id=?", (rid,))
                if row:
                    assets = fetch_all("SELECT id, asset_code, name FROM assets")
                    ac = [f"{a[1]} — {a[2]} (ID:{a[0]})" for a in assets]
                    cur = row[0]
                    aid_str = next((f"{a[1]} — {a[2]} (ID:{a[0]})" for a in assets if a[0]==cur[0]), "")
                    defaults = [aid_str] + list(cur[1:])
                    self._dialog("EDIT PM SCHEDULE", self._pm_fields(defaults, ac), lambda v: self._update_pm(rid, v), height=620)
        def _delete():
            rid = self._get_selected_id("pm")
            if rid:
                self._confirm_delete("Delete this PM schedule?",
                    lambda: [run_query("DELETE FROM pm_schedules WHERE id=?", (rid,)), self.show_pm()])
        def _complete():
            rid = self._get_selected_id("pm")
            if rid:
                today = datetime.date.today().isoformat()
                row = fetch_all("SELECT frequency_type, frequency_value FROM pm_schedules WHERE id=?", (rid,))[0]
                nd = self._calc_next_due(row[0], row[1])
                run_query("UPDATE pm_schedules SET last_completion_date=?, next_due_date=? WHERE id=?",
                          (today, nd, rid))
                messagebox.showinfo("PM Completed", f"Marked complete. Next due: {nd}")
                self.show_pm()

        card = self._section_card(frame, "")
        card.pack(fill="both", expand=True, padx=18, pady=6)
        self._toolbar_compact(card, _add, _edit, _delete,
                              extra_btns=[("✔ MARK COMPLETE", _complete, C["success"])])
        data = fetch_all("""
            SELECT pm.id, a.asset_code, a.criticality, pm.task_name, pm.frequency_type,
                   pm.frequency_value, pm.next_due_date, pm.last_completion_date,
                   pm.assigned_team, pm.estimated_duration_hrs, pm.status
            FROM pm_schedules pm JOIN assets a ON pm.asset_id=a.id
            ORDER BY pm.next_due_date
        """)
        hdrs = ["Asset", "Criticality", "Task Name", "Freq", "Value", "Next Due", "Last Done", "Team", "Est.Hrs", "Status"]
        self._build_table(card, hdrs, data, col_widths=[75, 80, 220, 80, 60, 100, 100, 130, 70, 75], compact=True)

    def _pm_fields(self, d=None, asset_choices=None):
        d = d or [""]*12
        ac = asset_choices or []
        return [
            {"label": "Asset *",                "key": "asset",    "type": "combobox", "values": ac,    "required": True, "default": d[0]},
            {"label": "Task Name *",             "key": "task",     "type": "entry",                     "required": True, "default": d[1]},
            {"label": "Frequency Type",          "key": "ftype",    "type": "combobox", "values": ["daily","weekly","monthly","quarterly","yearly"], "default": d[2]},
            {"label": "Frequency Value",         "key": "fval",     "type": "int",                       "default": d[3]},
            {"label": "Last Completion Date",    "key": "ldate",    "type": "entry",                     "default": d[4]},
            {"label": "Estimated Duration (hrs)","key": "est",      "type": "float",                     "default": d[6]},
            {"label": "Assigned Team",           "key": "team",     "type": "entry",                     "default": d[7]},
            {"label": "Procedure Reference",     "key": "proc",     "type": "entry",                     "default": d[8]},
            {"label": "Checklist Notes",         "key": "checklist","type": "text",                      "default": d[9]},
            {"label": "Status",                  "key": "status",   "type": "combobox", "values": ["Active","Suspended","Completed"], "default": d[10] or "Active"},
        ]

    def _calc_next_due(self, freq_type, freq_val):
        freq_val = int(freq_val) if freq_val else 1
        today = datetime.date.today()
        if   freq_type == "daily":     return (today + datetime.timedelta(days=freq_val)).isoformat()
        elif freq_type == "weekly":    return (today + datetime.timedelta(weeks=freq_val)).isoformat()
        elif freq_type == "monthly":
            m = today.month + freq_val
            y = today.year + (m-1)//12
            m = ((m-1)%12)+1
            import calendar
            d = min(today.day, calendar.monthrange(y, m)[1])
            return datetime.date(y, m, d).isoformat()
        elif freq_type == "quarterly": return (today + datetime.timedelta(days=90*freq_val)).isoformat()
        else:                          return today.replace(year=today.year+freq_val).isoformat()

    def _insert_pm(self, v):
        m = re.search(r'ID:(\d+)', v["asset"])
        if not m:
            raise ValueError("Invalid asset selection")
        asset_id = int(m.group(1))
        nd = self._calc_next_due(v["ftype"], v["fval"])
        run_query("INSERT INTO pm_schedules (asset_id,task_name,frequency_type,frequency_value,last_completion_date,next_due_date,estimated_duration_hrs,assigned_team,procedure,checklist,status) VALUES (?,?,?,?,?,?,?,?,?,?,?)",
                  (asset_id, v["task"], v["ftype"], v["fval"], v["ldate"], nd,
                   v["est"], v["team"], v["proc"], v["checklist"], v["status"] or "Active"))
        self.show_pm()

    def _update_pm(self, pid, v):
        m = re.search(r'ID:(\d+)', v["asset"])
        if not m:
            raise ValueError("Invalid asset selection")
        asset_id = int(m.group(1))
        nd = self._calc_next_due(v["ftype"], v["fval"])
        run_query("UPDATE pm_schedules SET asset_id=?,task_name=?,frequency_type=?,frequency_value=?,last_completion_date=?,next_due_date=?,estimated_duration_hrs=?,assigned_team=?,procedure=?,checklist=?,status=? WHERE id=?",
                  (asset_id, v["task"], v["ftype"], v["fval"], v["ldate"], nd,
                   v["est"], v["team"], v["proc"], v["checklist"], v["status"], pid))
        self.show_pm()

    # ══════════════════════════════════════════════════════════════════════════
    #  WORK ORDERS (compact toolbar + compact table)
    # ══════════════════════════════════════════════════════════════════════════
    def show_wo(self):
        self._set_page(self._build_wo, "WORK ORDERS")

    def _build_wo(self, frame):
        self._page_header(frame, "WORK ORDERS", "Corrective · Preventive · Emergency")

        strip = ctk.CTkFrame(frame, fg_color="transparent")
        strip.pack(fill="x", padx=18, pady=4)
        for col, (st, color) in enumerate([
            ("Open",        C["accent2"]),
            ("In Progress", C["warning"]),
            ("On Hold",     "#8B5CF6"),
            ("Completed",   C["success"]),
            ("Cancelled",   C["muted"]),
        ]):
            n = fetch_all("SELECT COUNT(*) FROM work_orders WHERE status=?", (st,))[0][0]
            self._kpi_card(strip, st.upper(), n, color=color, col=col)

        def _add():
            assets = fetch_all("SELECT id, asset_code, name FROM assets ORDER BY asset_code")
            ac = [f"{a[1]} — {a[2]} (ID:{a[0]})" for a in assets]
            self._dialog("NEW WORK ORDER", self._wo_fields(asset_choices=ac), self._insert_wo, height=760)
        def _edit():
            rid = self._get_selected_id("wo")
            if rid:
                row = fetch_all("SELECT wo_number,asset_id,wo_type,title,description,priority,criticality,status,reported_by,assigned_to,scheduled_date,estimated_hrs,actual_hrs,downtime_hrs,labor_cost,parts_cost,failure_mode,root_cause,corrective_action,notes FROM work_orders WHERE id=?", (rid,))
                if row:
                    assets = fetch_all("SELECT id, asset_code, name FROM assets")
                    ac = [f"{a[1]} — {a[2]} (ID:{a[0]})" for a in assets]
                    cur = row[0]
                    aid_str = next((f"{a[1]} — {a[2]} (ID:{a[0]})" for a in assets if a[0]==cur[1]), "")
                    d = [cur[0], aid_str] + list(cur[2:])
                    self._dialog("EDIT WORK ORDER", self._wo_fields(d, ac), lambda v: self._update_wo(rid, v), height=760)
        def _delete():
            rid = self._get_selected_id("wo")
            if rid:
                self._confirm_delete("Delete this work order permanently?",
                    lambda: [run_query("DELETE FROM work_orders WHERE id=?", (rid,)), self.show_wo()])
        def _close():
            rid = self._get_selected_id("wo")
            if rid:
                today = datetime.date.today().isoformat()
                run_query("UPDATE work_orders SET status='Closed', closed_date=? WHERE id=?", (today, rid))
                messagebox.showinfo("Closed", "Work Order closed.")
                self.show_wo()
        def _complete():
            rid = self._get_selected_id("wo")
            if rid:
                today = datetime.date.today().isoformat()
                run_query("UPDATE work_orders SET status='Completed', completed_date=? WHERE id=?", (today, rid))
                messagebox.showinfo("Completed", "Work Order marked as Completed.")
                self.show_wo()

        card = self._section_card(frame, "")
        card.pack(fill="both", expand=True, padx=18, pady=6)
        self._toolbar_compact(card, _add, _edit, _delete, extra_btns=[
            ("✔ COMPLETE", _complete, C["success"]),
            ("⬛ CLOSE",    _close,   "#546E7A"),
        ])
        data = fetch_all("""
            SELECT wo.id, wo.wo_number, a.asset_code, a.criticality, wo.wo_type,
                   wo.title, wo.priority, wo.status, wo.assigned_to, wo.scheduled_date,
                   wo.downtime_hrs, (wo.labor_cost+wo.parts_cost)
            FROM work_orders wo JOIN assets a ON wo.asset_id=a.id
            ORDER BY
                CASE wo.priority WHEN 'Critical' THEN 1 WHEN 'High' THEN 2 WHEN 'Medium' THEN 3 ELSE 4 END,
                wo.id DESC
        """)
        hdrs = ["WO #", "Asset", "A.Crit", "Type", "Title", "Priority", "Status", "Assigned", "Scheduled", "Downtime(h)", "Cost($)"]
        self._build_table(card, hdrs, data,
                          col_widths=[100, 75, 70, 55, 200, 70, 90, 110, 95, 90, 80],
                          highlight_col=5,
                          highlight_map={s: f"status_{s}" for s in STATUS_COLORS},
                          compact=True)

    def _wo_fields(self, d=None, asset_choices=None):
        d = d or [""]*20
        ac = asset_choices or []
        return [
            {"label": "WO Number *",            "key": "wo_num",   "type": "entry",    "required": True, "default": d[0], "placeholder": "e.g. WO-2025-009"},
            {"label": "Asset *",                 "key": "asset",    "type": "combobox", "values": ac,     "required": True, "default": d[1]},
            {"label": "WO Type",                 "key": "wo_type",  "type": "combobox", "values": ["PM","CM","EM","Inspection","Project"], "default": d[2]},
            {"label": "Title *",                 "key": "title",    "type": "entry",    "required": True, "default": d[3]},
            {"label": "Description",             "key": "desc",     "type": "text",     "default": d[4]},
            {"label": "Priority",                "key": "priority", "type": "combobox", "values": ["Critical","High","Medium","Low"], "default": d[5] or "Medium"},
            {"label": "Asset Criticality",       "key": "crit",     "type": "combobox", "values": ["Critical","Major","Minor"], "default": d[6] or "Minor"},
            {"label": "Status",                  "key": "status",   "type": "combobox", "values": ["Open","In Progress","On Hold","Completed","Closed","Cancelled"], "default": d[7] or "Open"},
            {"label": "Reported By",             "key": "rep_by",   "type": "entry",    "default": d[8]},
            {"label": "Assigned To",             "key": "asgn",     "type": "entry",    "default": d[9]},
            {"label": "Scheduled Date (YYYY-MM-DD)", "key": "sdate","type": "entry",    "default": d[10]},
            {"label": "Estimated Hours",         "key": "est_hrs",  "type": "float",    "default": d[11]},
            {"label": "Actual Hours",            "key": "act_hrs",  "type": "float",    "default": d[12]},
            {"label": "Downtime Hours",          "key": "dtime",    "type": "float",    "default": d[13]},
            {"label": "Labor Cost ($)",          "key": "lcost",    "type": "float",    "default": d[14]},
            {"label": "Parts Cost ($)",          "key": "pcost",    "type": "float",    "default": d[15]},
            {"label": "Failure Mode",            "key": "fmode",    "type": "entry",    "default": d[16]},
            {"label": "Root Cause",              "key": "rcause",   "type": "text",     "default": d[17]},
            {"label": "Corrective Action",       "key": "caction",  "type": "text",     "default": d[18]},
            {"label": "Notes",                   "key": "notes",    "type": "text",     "default": d[19]},
        ]

    def _insert_wo(self, v):
        m = re.search(r'ID:(\d+)', v["asset"])
        if not m:
            raise ValueError("Invalid asset selection")
        asset_id = int(m.group(1))
        today = datetime.date.today().isoformat()
        sched = v["sdate"] or today
        run_query("""INSERT INTO work_orders
            (wo_number,asset_id,wo_type,title,description,priority,criticality,status,
             reported_date,scheduled_date,reported_by,assigned_to,
             estimated_hrs,actual_hrs,downtime_hrs,labor_cost,parts_cost,
             failure_mode,root_cause,corrective_action,notes)
            VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)""",
            (v["wo_num"], asset_id, v["wo_type"], v["title"], v["desc"],
             v["priority"], v["crit"], v["status"], today, sched,
             v["rep_by"], v["asgn"],
             v["est_hrs"], v["act_hrs"], v["dtime"] or 0,
             v["lcost"] or 0, v["pcost"] or 0,
             v["fmode"], v["rcause"], v["caction"], v["notes"]))
        self.show_wo()

    def _update_wo(self, wid, v):
        m = re.search(r'ID:(\d+)', v["asset"])
        if not m:
            raise ValueError("Invalid asset selection")
        asset_id = int(m.group(1))
        run_query("""UPDATE work_orders SET
            wo_number=?,asset_id=?,wo_type=?,title=?,description=?,priority=?,criticality=?,status=?,
            scheduled_date=?,reported_by=?,assigned_to=?,
            estimated_hrs=?,actual_hrs=?,downtime_hrs=?,labor_cost=?,parts_cost=?,
            failure_mode=?,root_cause=?,corrective_action=?,notes=?
            WHERE id=?""",
            (v["wo_num"], asset_id, v["wo_type"], v["title"], v["desc"],
             v["priority"], v["crit"], v["status"],
             v["sdate"], v["rep_by"], v["asgn"],
             v["est_hrs"], v["act_hrs"], v["dtime"] or 0,
             v["lcost"] or 0, v["pcost"] or 0,
             v["fmode"], v["rcause"], v["caction"], v["notes"], wid))
        self.show_wo()

    # ══════════════════════════════════════════════════════════════════════════
    #  INVENTORY (compact toolbar + compact table)
    # ══════════════════════════════════════════════════════════════════════════
    def show_inventory(self):
        self._set_page(self._build_inventory, "INVENTORY")

    def _build_inventory(self, frame):
        self._page_header(frame, "SPARE PARTS INVENTORY", "Warehouse & Storeroom Management")

        strip = ctk.CTkFrame(frame, fg_color="transparent")
        strip.pack(fill="x", padx=18, pady=4)
        total_parts = fetch_all("SELECT COUNT(*) FROM inventory")[0][0]
        low_stock   = fetch_all("SELECT COUNT(*) FROM inventory WHERE quantity <= min_stock")[0][0]
        total_val   = fetch_all("SELECT COALESCE(SUM(quantity*unit_price),0) FROM inventory")[0][0]
        zero_stock  = fetch_all("SELECT COUNT(*) FROM inventory WHERE quantity=0")[0][0]
        self._kpi_card(strip, "TOTAL PARTS",     total_parts, color=C["text"],    col=0)
        self._kpi_card(strip, "LOW STOCK ALERT", low_stock,   color=C["danger"],  col=1)
        self._kpi_card(strip, "ZERO STOCK",      zero_stock,  color=C["danger"],  col=2)
        self._kpi_card(strip, "STOCK VALUE ($)", f"{total_val:,.0f}", color=C["success"], col=3)

        def _add():
            self._dialog("ADD SPARE PART", self._inv_fields(), self._insert_inv, height=680)
        def _edit():
            rid = self._get_selected_id("inv")
            if rid:
                row = fetch_all("SELECT part_number,part_name,category,quantity,reserved_qty,min_stock,max_stock,reorder_point,unit,location,bin_number,supplier,lead_time_days,unit_price,notes FROM inventory WHERE id=?", (rid,))
                if row:
                    self._dialog("EDIT SPARE PART", self._inv_fields(row[0]), lambda v: self._update_inv(rid, v), height=680)
        def _delete():
            rid = self._get_selected_id("inv")
            if rid:
                self._confirm_delete("Delete this inventory item?",
                    lambda: [run_query("DELETE FROM inventory WHERE id=?", (rid,)), self.show_inventory()])
        def _receive():
            rid = self._get_selected_id("inv")
            if rid:
                dlg = ctk.CTkInputDialog(text="Enter quantity received:", title="Receive Stock")
                qty_str = dlg.get_input()
                if qty_str:
                    try:
                        qty = int(qty_str)
                        run_query("UPDATE inventory SET quantity=quantity+?, last_received_date=? WHERE id=?",
                                  (qty, datetime.date.today().isoformat(), rid))
                        messagebox.showinfo("Stock Updated", f"+{qty} units added.")
                        self.show_inventory()
                    except ValueError:
                        messagebox.showerror("Error", "Enter a valid integer quantity.")

        card = self._section_card(frame, "")
        card.pack(fill="both", expand=True, padx=18, pady=6)
        self._toolbar_compact(card, _add, _edit, _delete,
                              extra_btns=[("▲ RECEIVE", _receive, C["accent2"])])
        data = fetch_all("""
            SELECT id, part_number, part_name, category, quantity, reserved_qty,
                   min_stock, reorder_point, unit, location, bin_number, unit_price,
                   CASE WHEN quantity<=0 THEN 'OUT' WHEN quantity<=min_stock THEN 'LOW' ELSE 'OK' END as stock_status
            FROM inventory ORDER BY
                CASE WHEN quantity<=0 THEN 0 WHEN quantity<=min_stock THEN 1 ELSE 2 END, part_name
        """)
        hdrs = ["Part #", "Name", "Category", "Qty", "Reserved", "Min", "ReorderPt", "Unit", "Location", "Bin", "Price($)", "Stock"]
        self._build_table(card, hdrs, data,
                          col_widths=[100, 180, 90, 50, 65, 50, 70, 50, 100, 70, 75, 60],
                          compact=True)

    def _inv_fields(self, d=None):
        d = d or [""]*15
        return [
            {"label": "Part Number *",   "key": "pnum",   "type": "entry", "required": True, "default": d[0]},
            {"label": "Part Name *",     "key": "pname",  "type": "entry", "required": True, "default": d[1]},
            {"label": "Category",        "key": "cat",    "type": "combobox", "values": ["Bearings","Seals","Lubricants","Gaskets","Filters","Belts","Couplings","Electrical","Instrumentation","Fasteners","Valves","Pipes & Fittings","Other"], "default": d[2]},
            {"label": "Quantity *",      "key": "qty",    "type": "int",   "required": True, "default": d[3]},
            {"label": "Reserved Qty",    "key": "rsv",    "type": "int",   "default": d[4]},
            {"label": "Min Stock *",     "key": "min",    "type": "int",   "required": True, "default": d[5]},
            {"label": "Max Stock",       "key": "max",    "type": "int",   "default": d[6]},
            {"label": "Reorder Point",   "key": "rpt",    "type": "int",   "default": d[7]},
            {"label": "Unit",            "key": "unit",   "type": "combobox", "values": ["pcs","m","L","kg","set","roll","box"], "default": d[8] or "pcs"},
            {"label": "Location",        "key": "loc",    "type": "entry", "default": d[9]},
            {"label": "Bin Number",      "key": "bin",    "type": "entry", "default": d[10]},
            {"label": "Supplier",        "key": "supp",   "type": "entry", "default": d[11]},
            {"label": "Lead Time (days)","key": "lead",   "type": "int",   "default": d[12]},
            {"label": "Unit Price ($)",  "key": "price",  "type": "float", "default": d[13]},
            {"label": "Notes",           "key": "notes",  "type": "text",  "default": d[14]},
        ]

    def _insert_inv(self, v):
        run_query("INSERT INTO inventory (part_number,part_name,category,quantity,reserved_qty,min_stock,max_stock,reorder_point,unit,location,bin_number,supplier,lead_time_days,unit_price,notes) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
                  (v["pnum"],v["pname"],v["cat"],v["qty"],v["rsv"],v["min"],v["max"],v["rpt"],
                   v["unit"],v["loc"],v["bin"],v["supp"],v["lead"],v["price"],v["notes"]))
        self.show_inventory()

    def _update_inv(self, iid, v):
        run_query("UPDATE inventory SET part_number=?,part_name=?,category=?,quantity=?,reserved_qty=?,min_stock=?,max_stock=?,reorder_point=?,unit=?,location=?,bin_number=?,supplier=?,lead_time_days=?,unit_price=?,notes=? WHERE id=?",
                  (v["pnum"],v["pname"],v["cat"],v["qty"],v["rsv"],v["min"],v["max"],v["rpt"],
                   v["unit"],v["loc"],v["bin"],v["supp"],v["lead"],v["price"],v["notes"],iid))
        self.show_inventory()

    # ══════════════════════════════════════════════════════════════════════════
    #  LABOR (unchanged)
    # ══════════════════════════════════════════════════════════════════════════
    def show_labor(self):
        self._set_page(self._build_labor, "LABOR")

    def _build_labor(self, frame):
        self._page_header(frame, "LABOR & PERSONNEL", "Maintenance Workforce Registry")

        strip = ctk.CTkFrame(frame, fg_color="transparent")
        strip.pack(fill="x", padx=18, pady=4)
        total   = fetch_all("SELECT COUNT(*) FROM labor")[0][0]
        avail   = fetch_all("SELECT COUNT(*) FROM labor WHERE available=1")[0][0]
        avg_rate= fetch_all("SELECT ROUND(AVG(hourly_rate),2) FROM labor")[0][0] or 0
        self._kpi_card(strip, "TOTAL PERSONNEL", total,           color=C["text"],    col=0)
        self._kpi_card(strip, "AVAILABLE",        avail,           color=C["success"], col=1)
        self._kpi_card(strip, "AVG RATE ($/hr)",  f"{avg_rate:.2f}", color=C["accent"], col=2)

        def _add():
            self._dialog("ADD PERSONNEL", self._labor_fields(), self._insert_labor, height=580)
        def _edit():
            rid = self._get_selected_id("labor")
            if rid:
                row = fetch_all("SELECT employee_id,name,skill,trade,shift,hourly_rate,certification,available FROM labor WHERE id=?", (rid,))
                if row:
                    self._dialog("EDIT PERSONNEL", self._labor_fields(row[0]), lambda v: self._update_labor(rid, v), height=580)
        def _delete():
            rid = self._get_selected_id("labor")
            if rid:
                self._confirm_delete("Remove this personnel record?",
                    lambda: [run_query("DELETE FROM labor WHERE id=?", (rid,)), self.show_labor()])

        card = self._section_card(frame, "")
        card.pack(fill="both", expand=True, padx=18, pady=6)
        self._toolbar(card, _add, _edit, _delete)
        data = fetch_all("SELECT id, employee_id, name, skill, trade, shift, hourly_rate, certification, CASE available WHEN 1 THEN 'Available' ELSE 'Unavailable' END FROM labor ORDER BY trade, name")
        hdrs = ["Emp ID", "Name", "Skill", "Trade", "Shift", "Rate($/hr)", "Certification", "Availability"]
        self._build_table(card, hdrs, data, col_widths=[80, 160, 150, 110, 70, 80, 180, 95])

    def _labor_fields(self, d=None):
        d = d or [""]*8
        return [
            {"label": "Employee ID",   "key": "empid", "type": "entry",    "default": d[0]},
            {"label": "Full Name *",   "key": "name",  "type": "entry",    "required": True, "default": d[1]},
            {"label": "Skill",         "key": "skill", "type": "entry",    "default": d[2]},
            {"label": "Trade",         "key": "trade", "type": "combobox", "values": ["Mechanical","Electrical","I&C","Operations","HSE","Civil","Other"], "default": d[3]},
            {"label": "Shift",         "key": "shift", "type": "combobox", "values": ["Day","Night","Shift","Rotating"], "default": d[4]},
            {"label": "Hourly Rate ($)","key": "rate", "type": "float",    "default": d[5]},
            {"label": "Certifications","key": "cert",  "type": "entry",    "default": d[6]},
            {"label": "Available",     "key": "avail", "type": "combobox", "values": ["Yes","No"], "default": "Yes" if str(d[7]) == "1" else "No"},
        ]

    def _insert_labor(self, v):
        avail = 1 if v["avail"] == "Yes" else 0
        run_query("INSERT INTO labor (employee_id,name,skill,trade,shift,hourly_rate,certification,available) VALUES (?,?,?,?,?,?,?,?)",
                  (v["empid"],v["name"],v["skill"],v["trade"],v["shift"],v["rate"],v["cert"],avail))
        self.show_labor()

    def _update_labor(self, lid, v):
        avail = 1 if v["avail"] == "Yes" else 0
        run_query("UPDATE labor SET employee_id=?,name=?,skill=?,trade=?,shift=?,hourly_rate=?,certification=?,available=? WHERE id=?",
                  (v["empid"],v["name"],v["skill"],v["trade"],v["shift"],v["rate"],v["cert"],avail,lid))
        self.show_labor()

    # ══════════════════════════════════════════════════════════════════════════
    #  REPORTS (unchanged)
    # ══════════════════════════════════════════════════════════════════════════
    def show_reports(self):
        self._set_page(self._build_reports, "REPORTS")

    def _build_reports(self, frame):
        self._page_header(frame, "MANAGEMENT REPORTS", "KPIs · Compliance · Analysis")

        # ── KPI Summary ────────────────────────────────────────────────────────
        kpi_card = self._section_card(frame, "IE MAINTENANCE KPIs")
        kpi_card.pack(fill="x", padx=18, pady=6)
        kpi_inner = ctk.CTkFrame(kpi_card, fg_color="transparent")
        kpi_inner.pack(fill="x", padx=10, pady=8)

        total_wos     = fetch_all("SELECT COUNT(*) FROM work_orders")[0][0]
        completed_wos = fetch_all("SELECT COUNT(*) FROM work_orders WHERE status='Completed'")[0][0]
        total_downtime= fetch_all("SELECT COALESCE(SUM(downtime_hrs),0) FROM work_orders")[0][0]
        avg_mttr      = fetch_all("SELECT ROUND(AVG(actual_hrs),1) FROM work_orders WHERE status='Completed' AND actual_hrs IS NOT NULL")[0][0] or 0
        total_cost    = fetch_all("SELECT COALESCE(SUM(labor_cost+parts_cost),0) FROM work_orders")[0][0]
        wo_completion = round(completed_wos / total_wos * 100, 1) if total_wos else 0

        kpis = [
            ("WO COMPLETION %",    f"{wo_completion}%", C["success"]),
            ("TOTAL DOWNTIME",     f"{total_downtime:.0f}h", C["danger"]),
            ("AVG MTTR",           f"{avg_mttr}h", C["warning"]),
            ("TOTAL MAINT. COST",  f"${total_cost:,.0f}", C["accent2"]),
            ("PM BACKLOG",         fetch_all("SELECT COUNT(*) FROM pm_schedules WHERE next_due_date < date('now')")[0][0], C["danger"]),
        ]
        for col, (t, v, c) in enumerate(kpis):
            self._kpi_card(kpi_inner, t, v, color=c, col=col)

        # ── Report buttons ─────────────────────────────────────────────────────
        actions_card = self._section_card(frame, "GENERATE REPORTS")
        actions_card.pack(fill="x", padx=18, pady=6)
        btn_grid = ctk.CTkFrame(actions_card, fg_color="transparent")
        btn_grid.pack(fill="x", padx=16, pady=12)
        for i in range(3):
            btn_grid.grid_columnconfigure(i, weight=1)

        reports = [
            ("COMPLETED WOs — THIS MONTH",  self._rpt_completed_month,  0, 0, C["success"]),
            ("OVERDUE PM SCHEDULES",         self._rpt_overdue_pm,       0, 1, C["danger"]),
            ("LOW / ZERO STOCK ITEMS",       self._rpt_low_stock,        0, 2, C["warning"]),
            ("CRITICAL ASSET STATUS",        self._rpt_critical_assets,  1, 0, C["danger"]),
            ("WO COST SUMMARY",              self._rpt_cost_summary,     1, 1, C["accent2"]),
            ("DOWNTIME ANALYSIS",            self._rpt_downtime,         1, 2, C["accent"]),
        ]
        btn_cfg = dict(font=self.F["h3"], corner_radius=8, height=42, text_color="#FFFFFF")
        for label, fn, row, col, color in reports:
            ctk.CTkButton(btn_grid, text=label, fg_color=color, hover_color=color,
                          command=fn, **btn_cfg).grid(row=row, column=col, padx=6, pady=6, sticky="ew")

        # ── Maintenance Type split chart ───────────────────────────────────────
        chart_row = ctk.CTkFrame(frame, fg_color="transparent")
        chart_row.pack(fill="x", padx=18, pady=6)

        c1 = self._section_card(chart_row, "WO TYPE SPLIT (PM / CM / EM)")
        c1.pack(side="left", expand=True, fill="both", padx=6, pady=6)
        self._chart_wo_type_split(c1)

        c2 = self._section_card(chart_row, "MONTHLY WORK ORDER TREND")
        c2.pack(side="left", expand=True, fill="both", padx=6, pady=6)
        self._chart_monthly_trend(c2)

    def _chart_wo_type_split(self, parent):
        data = fetch_all("SELECT wo_type, COUNT(*) FROM work_orders WHERE wo_type IS NOT NULL GROUP BY wo_type ORDER BY COUNT(*) DESC")
        if not data:
            ctk.CTkLabel(parent, text="No data yet", font=self.F["body"], text_color=C["muted"]).pack(pady=30)
            return
        labels = [r[0] for r in data]
        sizes  = [r[1] for r in data]
        type_colors = {"PM": "#2E7D32", "CM": "#E65100", "EM": "#C62828",
                       "Inspection": "#0057B8", "Project": "#6A1B9A"}
        colors_list = [type_colors.get(l, C["muted"]) for l in labels]
        fig, ax, canvas = self._make_fig(parent, (4.2, 2.8))
        fig.patch.set_facecolor("#FFFFFF")
        wedges, texts, autotexts = ax.pie(
            sizes, labels=None, autopct='%1.0f%%', startangle=140,
            colors=colors_list,
            wedgeprops={"linewidth": 2.5, "edgecolor": "#FFFFFF"},
            pctdistance=0.72,
        )
        for at in autotexts:
            at.set_color("#FFFFFF"); at.set_fontsize(9); at.set_fontweight("bold")
        legend_labels = [f"{l}  –  {s} WO" for l, s in zip(labels, sizes)]
        self._style_legend(ax, legend_labels)
        canvas.draw()

    def _chart_monthly_trend(self, parent):
        data = fetch_all("""
            SELECT strftime('%Y-%m', reported_date) as month, COUNT(*)
            FROM work_orders WHERE reported_date IS NOT NULL
            GROUP BY month ORDER BY month DESC LIMIT 6
        """)
        if not data:
            ctk.CTkLabel(parent, text="No trend data yet", font=self.F["body"], text_color=C["muted"]).pack(pady=30)
            return
        months = [r[0] for r in reversed(data)]
        counts = [r[1] for r in reversed(data)]
        fig, ax, canvas = self._make_fig(parent, (4.2, 2.8))
        fig.patch.set_facecolor("#FFFFFF")
        # Smooth area fill
        ax.fill_between(months, counts, alpha=0.12, color="#0057B8", zorder=1)
        ax.fill_between(months, counts, alpha=0.06, color="#0057B8", zorder=1)
        # Line + markers
        ax.plot(months, counts, color="#0057B8", linewidth=2.5,
                marker="o", markersize=7, markerfacecolor="#FFFFFF",
                markeredgecolor="#0057B8", markeredgewidth=2, zorder=3)
        # Value labels
        for x, y in zip(months, counts):
            ax.text(x, y + max(counts)*0.07, str(y),
                    ha="center", fontsize=8.5, fontweight="bold", color="#0057B8")
        self._style_ax(ax, ylabel="Work Orders")
        ax.tick_params(axis="x", rotation=30, labelsize=8)
        ax.set_ylim(0, max(counts) * 1.3 if counts else 1)
        canvas.draw()

    # ── Report popup functions ─────────────────────────────────────────────────
    def _rpt_completed_month(self):
        rows = fetch_all("""
            SELECT wo.wo_number, a.asset_code, wo.title, wo.completed_date,
                   COALESCE(wo.actual_hrs,0), COALESCE(wo.labor_cost+wo.parts_cost,0)
            FROM work_orders wo JOIN assets a ON wo.asset_id=a.id
            WHERE wo.status='Completed'
              AND strftime('%Y-%m', wo.completed_date)=strftime('%Y-%m','now')
        """)
        if not rows:
            messagebox.showinfo("Report", "No completed WOs this month.")
            return
        self._report_window("Completed WOs — This Month",
                            ["WO #", "Asset", "Title", "Completed", "Actual Hrs", "Total Cost($)"], rows)

    def _rpt_overdue_pm(self):
        rows = fetch_all("""
            SELECT pm.id, a.asset_code, a.criticality, pm.task_name, pm.next_due_date, pm.assigned_team
            FROM pm_schedules pm JOIN assets a ON pm.asset_id=a.id
            WHERE pm.next_due_date < date('now') ORDER BY a.criticality DESC
        """)
        if not rows:
            messagebox.showinfo("Report", "No overdue PM schedules. ✔")
            return
        self._report_window("Overdue PM Schedules", ["ID", "Asset", "Criticality", "Task", "Due Date", "Team"], rows)

    def _rpt_low_stock(self):
        rows = fetch_all("""
            SELECT id, part_number, part_name, quantity, min_stock, location,
                   ROUND((min_stock-quantity)*unit_price, 2)
            FROM inventory WHERE quantity <= min_stock ORDER BY (quantity-min_stock)
        """)
        if not rows:
            messagebox.showinfo("Report", "All stock levels are adequate. ✔")
            return
        self._report_window("Low / Zero Stock Items",
                            ["ID", "Part #", "Name", "Qty", "Min", "Location", "Reorder Value($)"], rows)

    def _rpt_critical_assets(self):
        rows = fetch_all("""
            SELECT a.id, a.asset_code, a.name, a.status, a.operating_hours,
                   a.mtbf_hours, a.mttr_hours,
                   COUNT(wo.id) as open_wos
            FROM assets a LEFT JOIN work_orders wo ON a.id=wo.asset_id
                AND wo.status NOT IN ('Completed','Closed','Cancelled')
            WHERE a.criticality='Critical' GROUP BY a.id ORDER BY a.status
        """)
        self._report_window("Critical Asset Status",
                            ["ID", "Code", "Name", "Status", "Op.Hrs", "MTBF", "MTTR", "Open WOs"], rows)

    def _rpt_cost_summary(self):
        rows = fetch_all("""
            SELECT a.asset_code, a.criticality,
                   COUNT(wo.id) as total_wos,
                   ROUND(COALESCE(SUM(wo.labor_cost),0),2) as labor,
                   ROUND(COALESCE(SUM(wo.parts_cost),0),2) as parts,
                   ROUND(COALESCE(SUM(wo.labor_cost+wo.parts_cost),0),2) as total
            FROM work_orders wo JOIN assets a ON wo.asset_id=a.id
            GROUP BY a.id ORDER BY total DESC
        """)
        self._report_window("Work Order Cost Summary by Asset",
                            ["Asset", "Criticality", "# WOs", "Labor($)", "Parts($)", "Total($)"], rows)

    def _rpt_downtime(self):
        rows = fetch_all("""
            SELECT a.asset_code, a.name, a.criticality,
                   COUNT(wo.id) as failures,
                   ROUND(COALESCE(SUM(wo.downtime_hrs),0),1) as total_dt,
                   ROUND(COALESCE(AVG(wo.downtime_hrs),0),1) as avg_dt
            FROM work_orders wo JOIN assets a ON wo.asset_id=a.id
            WHERE wo.wo_type IN ('CM','EM')
            GROUP BY a.id ORDER BY total_dt DESC
        """)
        self._report_window("Downtime Analysis (CM & EM WOs)",
                            ["Asset", "Name", "Criticality", "Failures", "Total DT(h)", "Avg DT(h)"], rows)

    def _report_window(self, title, headers, rows):
        """Popup report window with a table."""
        win = ctk.CTkToplevel(self.root)
        win.title(title)
        win.geometry("900x520")
        win.configure(fg_color=C["panel"])
        win.grab_set()

        ctk.CTkLabel(win, text=title, font=self.F["h2"], text_color=C["sidebar"]).pack(
            anchor="w", padx=20, pady=(14, 4))
        ctk.CTkFrame(win, height=1, fg_color=C["accent"]).pack(fill="x", padx=14)

        container = ctk.CTkFrame(win, fg_color="transparent")
        container.pack(fill="both", expand=True, padx=14, pady=10)

        # Wrap rows so _build_table gets (id, *cols) format
        wrapped = [(i+1, *r) for i, r in enumerate(rows)]
        self._build_table(container, headers, wrapped, col_widths=None)

        ctk.CTkButton(win, text="✕ CLOSE", fg_color=C["danger"], hover_color="#DC2626",
                      text_color="white", font=self.F["h3"], corner_radius=8, height=36,
                      command=win.destroy).pack(pady=10)


# ══════════════════════════════════════════════════════════════════════════════
#  ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    init_db()
    root = ctk.CTk()
    app = CMMSPro(root)
    root.mainloop()